# Burstiness e correlazioni a lungo raggio nei testi generati da LLM
### Replica del protocollo di Altmann, Cristadoro, Degli Esposti — *PNAS* 109:11582 (2012)

Il paper misura, su *Guerra e pace*, due sorgenti distinte di correlazioni a lungo raggio in
una sequenza binaria $x$ costruita da un testo:

1. la **burstiness**, cioè la coda larga della distribuzione $p(\tau)$ degli intervalli fra
   occorrenze successive dello stesso simbolo;
2. le **correlazioni fra quegli intervalli**, $C_\tau(k)$.

Il risultato centrale è che l'esponente $\gamma$ (da $\sigma_X^2(t)\sim t^\gamma$) si **conserva**
scendendo nella gerarchia topic → parole → lettere, mentre la burstiness **no**: le lettere sono
correlate a lungo raggio ma non bursty, i sostantivi chiave sono entrambe le cose.

Qui applichiamo lo stesso protocollo ai testi generati dai modelli Qwen, usando come riferimento
**lo stesso libro** del paper (`wrnpc`), così che i numeri siano confrontabili con le sue Fig. 2–4.

**La domanda.** Se le correlazioni a lungo raggio del testo umano nascono dalla persistenza
tematica ai livelli alti e colano verso il basso conservando $\gamma$ ma perdendo burstiness,
allora un modello che riproduca $\hat\gamma$ a livello di lettere **ma non** la burstiness dei
sostantivi sta imitando la statistica proiettata senza avere il meccanismo che la genera.
È una dissociazione falsificabile, ed è quello che questo notebook misura.

---

## Avvertenze metodologiche — leggerle prima dei risultati

**(A) Lo stimatore $\sigma_\tau/\langle\tau\rangle$ dipende dalla lunghezza del testo.**
Per code di potenza con $\mu < 3$ la varianza di $p(\tau)$ **diverge**: $\hat\sigma_\tau$ non
converge, cresce col numero di eventi. Il valore 3.86 che Altmann misura per *prince* su
3.2 M caratteri **non è confrontabile** con quello misurato su 42 k caratteri. Per questo tutti i
confronti qui sono contro **segmenti umani della stessa lunghezza**, mai contro il libro intero.
La §5 quantifica esplicitamente questa deriva.

**(B) Il prompt è Tolstoj.** I primi `PROMPT_CHARS` caratteri, se presenti nel campo
`generated_text`, non sono prodotti dal modello. Il notebook lo verifica e li rimuove.

**(C) I testi degenerati producono burstiness artefattuale.** Una parola in loop di ripetizione
dà $\sigma_\tau/\langle\tau\rangle$ enorme senza alcuna struttura semantica. Ogni documento
riceve una bandiera di degenerazione e le figure principali la usano per marcare i punti.

**(D) I testi ricuciti introducono una scala alle giunzioni.** Se `n_continuations > 0`, le
riprese si addensano dove $p(\text{EOS})$ ha il massimo, cioè proprio nella finestra
$10^2$–$10^3$ simboli dove Altmann colloca la transizione delle keyword. Il notebook correla
le metriche col numero di riprese e lo riporta.

**(E) Vantaggio rispetto al paper.** Altmann ha una sola realizzazione e media su finestre
scorrevoli. Qui ci sono decine di documenti indipendenti per cella. Attenzione però a come viene
sfruttato: il notebook stima $\hat\gamma$ **su ogni documento separatamente** e poi ne media i
valori; **non** media le curve $\sigma^2_X(t)$ prima del fit. Le due cose non coincidono, e in
tesi va riportata quella effettivamente calcolata (media dei $\hat\gamma$ con la sua deviazione
standard fra documenti). I testi **non** vengono mai concatenati (creerebbe intervalli spuri
alle giunzioni).

**Output**: `risultati_altmann/` (CSV + figure) e `risultati_altmann.zip`.

## 1. Configurazione

Unica cella da toccare.

In [ ]:
from pathlib import Path

# --- input / output ---
# I cinque dataset sweep_qwen*.jsonl stanno nella radice del repository: la si
# cerca risalendo dalla cartella corrente, cosi' il notebook funziona dovunque
# il repository sia stato clonato.
DATASET_GLOB = "sweep_qwen*.jsonl"
_qui = Path.cwd().resolve()
DATA_DIR = next((p for p in [_qui, *_qui.parents] if any(p.glob(DATASET_GLOB))), None)
if DATA_DIR is None:
    raise FileNotFoundError(f"Nessun file {DATASET_GLOB} trovato risalendo da {_qui}")

OUT_DIR   = Path("risultati_altmann")
ZIP_NAME  = "risultati_altmann"
CACHE_DIR = Path("corpora_cache")              # condivisa col notebook precedente

# --- corpus di riferimento (identico al notebook precedente) ---
GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/2600/pg2600.txt"   # Tolstoj, Guerra e pace
START_PHRASE  = "Well, Prince, so Genoa and Lucca"
PROMPT_CHARS  = 8262
# Copia locale opzionale del libro: se la rete non e' disponibile, mettere il .txt
# scaricato a mano da GUTENBERG_URL in uno di questi percorsi.
CORPUS_LOCALE = [DATA_DIR / "pg2600.txt", Path("pg2600.txt"), CACHE_DIR / "pg2600.txt"]

# --- lunghezze ---
ANALYSIS_CHARS  = 50_000   # troncamento comune a tutti i testi
N_ORIG_SEGMENTS = 20       # segmenti umani di riferimento, LUNGHI QUANTO I TESTI GENERATI
MIN_EVENTS      = 15       # sotto questo numero di occorrenze la sequenza non viene stimata
# Avvertenza (A): il confronto ha senso solo fra testi della STESSA lunghezza. Con questa
# bandiera i documenti piu' corti di ANALYSIS_CHARS vengono scartati invece di essere
# analizzati su una finestra piu' piccola (che ne abbasserebbe artificialmente cv_tau).
RICHIEDI_LUNGHEZZA_UGUALE = True

# --- stima delle correlazioni: sigma^2_X(t) ~ t^gamma ---
# Altmann fitta fino a t_s = 1% della lunghezza del libro. Su 50k caratteri l'1% sarebbe 500,
# troppo poco per vedere la transizione di paragrafo (10^2-10^3). Qui si allarga; il range e'
# esplicito e va riportato in tesi.
LAG_MIN, LAG_MAX = 4, 4000
N_LAGS           = 40
FIT_RANGE        = (100, 2000)     # [t_s', t_s] per il fit di gamma
FIT_RANGE_STRETTO = (100, 500)     # 1% della lunghezza: confronto conservativo alla Altmann

# --- selezione delle parole bersaglio ---
N_LETTERS      = 19    # lettere piu' frequenti (+ spazio + vocali)
N_FUNCTION     = 6     # parole piu' frequenti (function words)
N_KEYWORDS     = 7     # sostantivi / nomi propri
N_MATCHED      = 7     # controlli appaiati in frequenza (uno per keyword)
PROPER_CAP_RATIO = 0.6 # soglia per dichiarare nome proprio (maiuscolo non a inizio frase)

# --- moduli opzionali ---
RUN_FINITE_SIZE = True    # §5: deriva degli stimatori con la lunghezza (diagnostica chiave)
RUN_SHUFFLES_M1M2 = True  # §9: shuffling M1/M2 del paper
N_NULL_REPS     = 3       # ripetizioni dei null model A1/A2

# --- sequenze di cui conservare le curve complete (Fig. 2 e Fig. 4) ---
CURVE_LETTER  = "e"        # la lettera piu' frequente
CURVE_KEYWORD = "prince"   # se assente si usa la keyword piu' frequente disponibile

RANDOM_SEED = 20260811

# --- figure ---
REPRESENTATIVE_MODEL = "Qwen3.5-4B-Base"
DPI = 150

In [ ]:
import json, re, os, sys, math, ssl, shutil, random, hashlib, urllib.request, warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    from scipy import stats as sps
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

try:
    import certifi
    HAS_CERTIFI = True
except Exception:
    HAS_CERTIFI = False

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": DPI, "savefig.bbox": "tight",
    "font.size": 10, "axes.grid": True, "grid.alpha": 0.25,
    "axes.axisbelow": True, "legend.frameon": True, "legend.framealpha": 0.9,
})
warnings.filterwarnings("ignore", category=RuntimeWarning)

OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / "figure").mkdir(exist_ok=True)
(OUT_DIR / "dati").mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)
pyrng = random.Random(RANDOM_SEED)

LAGS = np.unique(np.logspace(np.log10(LAG_MIN), np.log10(LAG_MAX), N_LAGS).astype(int))
print("lag usati:", LAGS[:6], "...", LAGS[-3:], f"({len(LAGS)} valori)")
print("output ->", OUT_DIR.resolve())
print("scipy:", HAS_SCIPY, "| certifi:", HAS_CERTIFI)

## 2. Il nucleo: sequenze binarie, trasporto, intervalli, null model

Traduzione diretta delle equazioni del paper.

- **Eq. 1–2** — la sequenza binaria $x_k = f_\alpha(s_k^{k+r})$: 1 dove la condizione $\alpha$ è
  soddisfatta alla posizione $k$, 0 altrove. Il tempo è sempre contato **in caratteri**, a
  qualunque livello della gerarchia: è ciò che rende confrontabili lettere e parole.
- **Eq. 3** — $\sigma_X^2(t) = \langle X(t)^2\rangle - \langle X(t)\rangle^2 \simeq t^\gamma$,
  con $X(t)=\sum_{j\le t} x_j$ e la media su finestre scorrevoli. Si usa l'indicatore integrato
  invece di $C_x(t)$ perché la stima asintotica è molto più robusta.
- **$p(\tau)$** — gli intervalli fra 1 consecutivi; $\sigma_\tau/\langle\tau\rangle$ è
  l'indicatore di burstiness (1 per un processo di Poisson).
- **A1** — rimescola gli 0/1: distrugge tutto. Controllo che deve dare $\gamma \approx 1$.
- **A2** — rimescola gli intervalli $\tau_i$: distrugge $C_\tau(k)$ ma **preserva** $p(\tau)$.
  Se $\hat\gamma_{A2} \approx \hat\gamma$ l'origine è la burstiness; se $\hat\gamma_{A2}\approx 1$
  mentre $\hat\gamma > 1$, l'origine sono le correlazioni fra intervalli.

In [ ]:
# --- tokenizzazione (unicode-aware) -------------------------------------------------
# Guerra e pace contiene Rostov, Natasha, Bolkonski con accenti: con una classe [A-Za-z]
# il tokenizzatore troncherebbe "Rostov" accentato in "rost", e la parola non verrebbe
# poi piu' ritrovata nel testo, perche' \b fra "t" e la vocale accentata non e' un
# confine di parola (la vocale accentata e' \w). Risultato: due delle sette keyword
# risultavano avere ZERO occorrenze pur essendo fra le piu' frequenti del libro.
WORD_CHAR = r"[^\W\d_]"                                     # lettera unicode, niente cifre/underscore
TOKEN_RE  = re.compile(WORD_CHAR + r"+(?:['\u2019\-]" + WORD_CHAR + r"+)*", re.UNICODE)


def build_char_array(text):
    '''array numpy di caratteri minuscoli: costruito una volta per documento'''
    return np.array(list(text.lower()))

def pos_from_char(carr, ch):
    return np.flatnonzero(carr == ch)

def pos_from_set(carr, chars):
    return np.flatnonzero(np.isin(carr, list(chars)))

_word_re_cache = {}
def pos_from_word(text, w):
    '''posizioni (in caratteri) di inizio di ogni occorrenza della parola w, case-insensitive.

    I confini usano la stessa classe di caratteri del tokenizzatore: "prince" non matcha
    dentro "princess", ma le parole accentate vengono trovate regolarmente.'''
    if w not in _word_re_cache:
        _word_re_cache[w] = re.compile(
            r"(?<!" + WORD_CHAR + r")" + re.escape(w) + r"(?!" + WORD_CHAR + r")",
            re.IGNORECASE | re.UNICODE)
    return np.fromiter((m.start() for m in _word_re_cache[w].finditer(text)), dtype=np.int64)


def seme(*parti):
    '''seme deterministico e riproducibile fra sessioni.

    hash() sulle stringhe e' randomizzato da PYTHONHASHSEED: usarlo per inizializzare
    l'RNG renderebbe i null model diversi a ogni riavvio del kernel, nonostante
    RANDOM_SEED sia fissato.'''
    h = hashlib.sha256("|".join(map(str, parti)).encode("utf-8")).hexdigest()
    return (int(h[:8], 16) ^ RANDOM_SEED) % (2 ** 32)


def transport_sigma2(pos, N, lags=None):
    '''sigma^2_X(t) mediata su tutte le finestre scorrevoli di ampiezza t.

    pos: posizioni degli 1. N: lunghezza della sequenza. Ritorna array su `lags`.
    Implementazione: X(i+t)-X(i) su tutti gli i, poi varianza. O(N) per lag.'''
    lags = LAGS if lags is None else lags
    x = np.zeros(N, dtype=np.float64)
    if len(pos):
        x[pos[pos < N]] = 1.0
    C = np.concatenate(([0.0], np.cumsum(x)))
    out = np.full(len(lags), np.nan)
    for i, t in enumerate(lags):
        if t >= N // 4:            # meno di 4 finestre indipendenti: non stimabile
            continue
        d = C[t:] - C[:-t]
        out[i] = d.var()
    return out


def fit_gamma(lags, s2, lo, hi, min_pts=5):
    '''pendenza log-log di sigma^2 vs t nel range [lo,hi], con errore standard.'''
    lags = np.asarray(lags, float); s2 = np.asarray(s2, float)
    m = np.isfinite(s2) & (s2 > 0) & (lags >= lo) & (lags <= hi)
    if m.sum() < min_pts:
        return np.nan, np.nan
    X, Y = np.log10(lags[m]), np.log10(s2[m])
    n = len(X)
    b, a = np.polyfit(X, Y, 1)
    resid = Y - (b * X + a)
    sxx = ((X - X.mean()) ** 2).sum()
    se = float(np.sqrt((resid ** 2).sum() / max(n - 2, 1) / sxx)) if sxx > 0 else np.nan
    return float(b), se


def local_gamma(lags, s2):
    '''esponente locale gamma(t) = d log sigma^2 / d log t  (Fig. 4 del paper)'''
    lags = np.asarray(lags, float); s2 = np.asarray(s2, float)
    m = np.isfinite(s2) & (s2 > 0)
    if m.sum() < 3:
        return np.array([]), np.array([])
    X, Y = np.log10(lags[m]), np.log10(s2[m])
    return lags[m], np.gradient(Y, X)


def interevent(pos):
    '''tau_i = distanze fra 1 consecutivi (in caratteri)'''
    return np.diff(np.asarray(pos, dtype=np.int64)) if len(pos) > 1 else np.array([], np.int64)


def burstiness_stats(tau):
    if len(tau) < 2:
        return dict(mean_tau=np.nan, sigma_tau=np.nan, cv_tau=np.nan, B_goh=np.nan)
    m, s = float(tau.mean()), float(tau.std(ddof=1))
    return dict(mean_tau=m, sigma_tau=s, cv_tau=s / m if m > 0 else np.nan,
                B_goh=(s - m) / (s + m) if (s + m) > 0 else np.nan)


def null_A1(pos, N, r):
    '''shuffle degli {0,1}: stesso numero di eventi, posizioni casuali. Distrugge tutto.'''
    M = len(pos)
    return np.sort(r.choice(N, size=min(M, N), replace=False)) if M else pos


def null_A2(pos, N, r):
    '''shuffle degli intervalli: preserva p(tau), distrugge C_tau(k).'''
    tau = interevent(pos)
    if len(tau) < 2:
        return pos
    tau = r.permutation(tau)
    new = np.concatenate(([pos[0]], pos[0] + np.cumsum(tau)))
    return new[new < N]

In [ ]:
def analyze_sequence(pos, N, label, level, r, n_null=N_NULL_REPS,
                     fit_range=FIT_RANGE, keep_curves=False):
    '''Analisi completa di una singola sequenza binaria: gamma, burstiness, null A1/A2.'''
    pos = np.asarray(pos, dtype=np.int64)
    pos = pos[pos < N]
    M = len(pos)
    rec = dict(sequenza=label, livello=level, n_eventi=M,
               frequenza=M / N if N else np.nan, n_chars=N)
    if M < MIN_EVENTS:
        rec.update(dict(gamma=np.nan, gamma_se=np.nan, gamma_stretto=np.nan,
                        gamma_A1=np.nan, gamma_A2=np.nan,
                        mean_tau=np.nan, sigma_tau=np.nan, cv_tau=np.nan, B_goh=np.nan,
                        quota_burstiness=np.nan, stimabile=False))
        return rec, None

    s2 = transport_sigma2(pos, N)
    g, se = fit_gamma(LAGS, s2, *fit_range)
    gs, ses = fit_gamma(LAGS, s2, *FIT_RANGE_STRETTO)
    rec.update(dict(gamma=g, gamma_se=se, gamma_stretto=gs, stimabile=True))
    rec.update(burstiness_stats(interevent(pos)))

    g1, g2, cv2 = [], [], []
    s2_1 = np.zeros_like(s2); s2_2 = np.zeros_like(s2)
    for _ in range(n_null):
        p1 = null_A1(pos, N, r)
        p2 = null_A2(pos, N, r)
        c1 = transport_sigma2(p1, N); c2 = transport_sigma2(p2, N)
        s2_1 += c1 / n_null; s2_2 += c2 / n_null
        g1.append(fit_gamma(LAGS, c1, *fit_range)[0])
        g2.append(fit_gamma(LAGS, c2, *fit_range)[0])
        cv2.append(burstiness_stats(interevent(p2))["cv_tau"])
    rec["gamma_A1"] = float(np.nanmean(g1)); rec["gamma_A1_sd"] = float(np.nanstd(g1))
    rec["gamma_A2"] = float(np.nanmean(g2)); rec["gamma_A2_sd"] = float(np.nanstd(g2))
    # controllo interno: A2 permuta i tau, quindi cv_tau_A2 deve coincidere con cv_tau
    rec["cv_tau_A2"] = float(np.nanmean(cv2))
    # frazione di gamma attribuibile alla burstiness (gamma_A2 e' un lower bound: gamma >= gamma_A2)
    # NB: era `g is not np.nan`, che e' un confronto di identita' sempre vero su un float.
    rec["quota_burstiness"] = ((rec["gamma_A2"] - 1) / (g - 1)) \
                              if (np.isfinite(g) and g > 1.02) else np.nan

    curves = None
    if keep_curves:
        curves = dict(lags=LAGS, s2=s2, s2_A1=s2_1, s2_A2=s2_2, tau=interevent(pos))
    return rec, curves

## 3. Caricamento dei testi generati

Stesso loader del notebook precedente. In più: **verifica e rimozione del prompt**.
Se `generated_text` contiene la coda del prompt di Tolstoj, quei caratteri vanno tolti,
altrimenti si sta misurando Tolstoj e non il modello.

In [ ]:
def normalize_text(t):
    return t.replace("\r\n", "\n").replace("\r", "\n")

META_KEYS = ["model", "model_family", "architecture", "temperature", "sample_id", "seed",
             "prompt_sha256_16", "prompt_length_chars", "prompt_length_tokens",
             "completion_tokens", "generated_chars", "top_p", "top_k", "repetition_penalty",
             "finish_reason", "n_continuations", "run_id"]

rows, TEXTS = [], []
bad_lines = []
for p in sorted(DATA_DIR.glob(DATASET_GLOB)):
    with open(p, encoding="utf-8") as fh:
        # una riga malformata non deve far perdere silenziosamente il resto del file
        for ln, line in enumerate(fh, 1):
            if not line.strip():
                continue
            try:
                d = json.loads(line)
            except json.JSONDecodeError as e:
                bad_lines.append((p.name, ln, str(e)))
                continue
            if not isinstance(d.get("generated_text"), str):
                bad_lines.append((p.name, ln, "generated_text mancante o non stringa"))
                continue
            r = {k: d.get(k) for k in META_KEYS}
            r["file"] = p.name
            r["text_index"] = len(TEXTS)
            TEXTS.append(normalize_text(d["generated_text"]))
            r["n_chars_full"] = len(TEXTS[-1])
            rows.append(r)

df = pd.DataFrame(rows)
df["model_short"] = df["model"].astype(str).str.split("/").str[-1]

def param_size(name):
    m = re.search(r"(\d+(?:\.\d+)?)B", str(name))
    return float(m.group(1)) if m else np.nan
df["params_B"] = df["model_short"].map(param_size)
MODELS = df.groupby("model_short")["params_B"].first().sort_values().index.tolist()

if bad_lines:
    print(f"RIGHE SCARTATE: {len(bad_lines)}")
    for f, ln, e in bad_lines[:10]:
        print(f"  {f}:{ln}  {e[:80]}")
print(df.groupby(["model_short", "temperature"]).size().unstack(fill_value=0))
print("\ncampioni totali:", len(df))

# il prompt di generazione e' lo stesso in tutti i file?
_h = df.groupby("prompt_sha256_16")["file"].unique()
if len(_h) > 1:
    print("\nATTENZIONE: i file non condividono lo stesso prompt di generazione:")
    for h, ff in _h.items():
        n = int((df["prompt_sha256_16"] == h).sum())
        nc = df.loc[df["prompt_sha256_16"] == h, "prompt_length_chars"].iloc[0]
        print(f"  sha={h}  prompt_length_chars={nc}  n={n}  file={list(ff)}")
    print("  -> i testi restano confrontabili (sono continuazioni della stessa scena),")
    print("     ma la differenza va dichiarata nel capitolo metodologico.")

In [ ]:
jsonl = sorted(DATA_DIR.glob(DATASET_GLOB))
print("DATA_DIR:", DATA_DIR.resolve())
print(f"file .jsonl trovati: {len(jsonl)}")
for p in jsonl:
    print(f"  {p.name:50s} {p.stat().st_size/1e6:7.1f} MB")
if not jsonl:
    print("\nATTENZIONE: nessun .jsonl. Contenuto della cartella:")
    for p in sorted(DATA_DIR.iterdir())[:20]:
        print("  ", p.name)

In [ ]:
# --- corpus di riferimento ---
GUT_START = re.compile(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
GUT_END   = re.compile(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
STRUCT = re.compile(
    r"^[ \t]*(?:"
    r"BOOK\s+[A-Z]+[^\n]*|CHAPTER\s+[IVXLCDM\d]+[^\n]*|"
    r"(?:FIRST|SECOND)\s+EPILOGUE[^\n]*|EPILOGUE[^\n]*|CONTENTS[^\n]*|"
    r"PART\s+[IVXLCDM\d]+[^\n]*|APPENDIX[^\n]*|\d+"
    r")[ \t]*$", re.M)


def contesto_ssl():
    '''Contesto TLS robusto su Windows.

    Su questa installazione (Python 3.11 di Anaconda ricompilato contro OpenSSL 3.5)
    ssl.create_default_context() esplode con

        SSLError: [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4035)

    perche' SSLContext.load_default_certs() carica i certificati dello store di Windows
    passandoli come cadata in formato DER, e quel percorso e' rotto in questa
    combinazione: falliscono TUTTI i certificati dello store, non uno solo. Si aggira
    passando esplicitamente il bundle PEM di certifi, che non tocca lo store di Windows.'''
    if HAS_CERTIFI:
        try:
            return ssl.create_default_context(cafile=certifi.where())
        except Exception:
            pass
    try:
        return ssl.create_default_context()          # installazioni sane
    except ssl.SSLError:
        pass
    ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)    # ultima spiaggia: nessuna CA utilizzabile
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    print("ATTENZIONE: nessun bundle di CA utilizzabile, verifica del certificato disattivata.\n"
          "            Rimedio pulito:  pip install --upgrade certifi")
    return ctx


def fetch(url, cache_dir=CACHE_DIR):
    """Scarica il corpus e lo mette in cache SENZA passare per il testo.

    ATTENZIONE (bug sottile, si manifesta solo dalla seconda esecuzione in poi e solo
    su Windows): Path.write_text apre il file in modalita' testo, quindi traduce ogni
    "\n" in os.linesep = "\r\n". Il testo di Gutenberg ha gia' terminatori "\r\n",
    quindi ogni riga veniva salvata come "\r\r\n"; alla rilettura le universal
    newlines la riconvertivano in "\n\n". Risultato: ogni a-capo diventava un a-capo
    doppio, il corpus si allungava di ~66k caratteri, ogni riga diventava un capoverso,
    e soprattutto il testo umano finiva ad avere una densita' di spazi bianchi diversa
    da quella dei testi generati -- un confronto in cui il tempo si conta in caratteri
    ne risente direttamente. Si evita leggendo e scrivendo in binario."""
    fn = cache_dir / (hashlib.sha256(url.encode()).hexdigest()[:12] + ".txt")
    if fn.exists():
        raw = fn.read_bytes().decode("utf-8", errors="replace")
        if "\r\r" in raw:      # cache scritta da una versione precedente del notebook
            print(f"cache corrotta (newline raddoppiati): la rigenero -> {fn}")
            fn.unlink()
        else:
            print(f"corpus letto dalla cache: {fn}")
            return raw
    for loc in CORPUS_LOCALE:                        # copia locale messa a mano
        if loc.is_file():
            print(f"corpus letto dalla copia locale: {loc}")
            raw = loc.read_bytes().decode("utf-8", errors="replace")
            fn.write_bytes(raw.encode("utf-8"))
            return raw
    req = urllib.request.Request(url, headers={"User-Agent": "research-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=60, context=contesto_ssl()) as resp:
            raw = resp.read().decode("utf-8", errors="replace")
    except Exception as e:
        raise RuntimeError(
            f"Download di {url} fallito ({type(e).__name__}: {e}).\n"
            f"Rimedio: scaricare il file a mano dal browser e salvarlo come\n"
            f"  {CORPUS_LOCALE[0]}\n"
            f"poi rieseguire questa cella.") from e
    fn.write_bytes(raw.encode("utf-8"))
    print(f"corpus scaricato e messo in cache: {fn}")
    return raw


def clean_literary(raw, start_phrase=None):
    raw = raw.replace("\r\n", "\n").replace("\r", "\n")
    m = GUT_START.search(raw)
    if m: raw = raw[m.end():]
    m = GUT_END.search(raw)
    if m: raw = raw[:m.start()]
    if start_phrase:
        i = raw.find(start_phrase)
        if i < 0:
            raise RuntimeError("START_PHRASE non trovata")
        raw = raw[i:]
    raw = STRUCT.sub("", raw)
    raw = re.sub(r"\n[ \t]+\n", "\n\n", raw)
    return re.sub(r"\n{3,}", "\n\n", raw).strip()

CORPUS = clean_literary(fetch(GUTENBERG_URL), START_PHRASE)
PROMPT = CORPUS[:PROMPT_CHARS]
BODY   = CORPUS[PROMPT_CHARS:]
print(f"corpus: {len(CORPUS):,} caratteri | corpo dopo il prompt: {len(BODY):,}")

# il PROMPT ricostruito qui coincide con quello davvero usato in generazione?
_sha = hashlib.sha256(PROMPT.encode("utf-8")).hexdigest()[:16]
_att = sorted(set(df["prompt_sha256_16"].dropna().unique()))
print(f"sha256(PROMPT)[:16] ricostruito = {_sha} | registrato nei dataset = {_att}")
if _att and _sha not in _att:
    print("  -> NON coincide: la pulizia del corpus fatta qui differisce da quella usata in\n"
          "     generazione. Non invalida l'analisi (generated_text non contiene il prompt,\n"
          "     vedi cella seguente), ma BODY inizia a qualche decina di caratteri di distanza\n"
          "     dal punto esatto in cui i modelli hanno ripreso. Va dichiarato in tesi.")

In [ ]:
# --- il campo generated_text contiene il prompt? ---
tail = PROMPT[-200:]
def strip_prompt(t):
    i = t.find(tail)
    if i >= 0:
        return t[i + len(tail):], True
    # fallback: confronto sui primi caratteri
    if t[:200] == PROMPT[:200]:
        return t[PROMPT_CHARS:], True
    return t, False

n_stripped = 0
for i, t in enumerate(TEXTS):
    TEXTS[i], did = strip_prompt(t)
    n_stripped += did
df["n_chars_netti"] = [len(TEXTS[i]) for i in df["text_index"]]
print(f"documenti in cui il prompt era incluso ed e' stato rimosso: {n_stripped}/{len(TEXTS)}")
if n_stripped == 0:
    print("  -> `generated_text` contiene solo la continuazione: nessuna rimozione necessaria.")
elif n_stripped < len(TEXTS):
    print("  -> ATTENZIONE: comportamento disomogeneo fra file, controllare la generazione.")

# --- lunghezza di analisi comune (avvertenza A) ---
N_EFF = ANALYSIS_CHARS
if not (df["n_chars_netti"] >= ANALYSIS_CHARS).any():
    # nessun documento arriva ad ANALYSIS_CHARS: si scende alla lunghezza comune massima
    N_EFF = int(df["n_chars_netti"].min())
    print(f"ATTENZIONE: nessun documento raggiunge ANALYSIS_CHARS={ANALYSIS_CHARS:,};"
          f" N_EFF abbassato a {N_EFF:,}.")
df["usabile"] = df["n_chars_netti"] >= N_EFF
n_corti = int((~df["usabile"]).sum())
print(f"campioni piu' corti di {N_EFF:,} caratteri: {n_corti}"
      + ("  -> ESCLUSI dall'analisi (RICHIEDI_LUNGHEZZA_UGUALE=True)"
         if RICHIEDI_LUNGHEZZA_UGUALE else
         "  -> analizzati su una finestra piu' corta: il loro cv_tau NON e' confrontabile"))
if n_corti:
    print(df.loc[~df["usabile"], ["file", "model_short", "temperature", "sample_id",
                                  "n_chars_netti"]].to_string(index=False))
print(f"lunghezza di analisi effettiva: {N_EFF:,} caratteri "
      f"({int(df['usabile'].sum())} documenti usabili su {len(df)})")

In [ ]:
# --- segmenti umani di riferimento, DELLA STESSA LUNGHEZZA (avvertenza A) ---
starts = np.linspace(0, max(len(BODY) - N_EFF, 0), N_ORIG_SEGMENTS).astype(int)
ORIG_SEGMENTS = [BODY[s:s + N_EFF] for s in starts]
print(f"{len(ORIG_SEGMENTS)} segmenti umani da {N_EFF:,} caratteri")

# --- bandiere di degenerazione (avvertenza C) ---
# Ci sono TRE modi di degenerare, e servono tre indicatori distinti perche' due di
# essi si mascherano a vicenda:
#   (1) ripetizione: il modello entra in loop. Unicita' dei 10-grammi bassissima.
#   (2) collasso dell'alfabeto: ad alta T il modello smette di scrivere in inglese e
#       produce un miscuglio di script. Qui l'unicita' dei 10-grammi e' ALTISSIMA
#       (~0.99), quindi il criterio (1) da solo dichiara questi testi sani.
#   (3) collasso del lessico: il testo resta in caratteri latini, con punteggiatura
#       plausibile, ma le parole non sono inglese ("rediacis", "unforthétanistus") o
#       sono markup ("nbsp", "picframe", "ndash"). Passa (1) e (2) indenne.
# Il caso (3) e' quello che conta di piu' qui, perche' la degenerazione e' PROGRESSIVA
# dentro il documento: certi testi partono come prosa e collassano a meta'. Mediando
# sull'intera finestra da N_EFF, un documento mezzo buono e mezzo rumore passa (1) e (2).
LAT_RE = re.compile(r"[A-Za-z \n.,;:!?'\"-]")
FRASE_RE = re.compile(r"[.!?][\"'\u2019\)\]]*(?:\s|$)")

# Vocabolario di riferimento: TIPI attestati nel corpus umano. La copertura si misura
# sui TOKEN, non sui tipi, cosi' la deriva tematica non penalizza: un testo su un altro
# argomento ma in inglese normale usa comunque the/and/was/said/man/house, tutti
# attestati. Una copertura di 0.73 significa che un token su quattro e' una parola mai
# usata in 3,2 milioni di caratteri di romanzo inglese, che e' difficile da giustificare
# come scelta stilistica.
VOC_REF = set(w.lower() for w in TOKEN_RE.findall(BODY))
print(f"vocabolario di riferimento: {len(VOC_REF):,} tipi")

def degeneracy_flags(t, k=10, step=3):
    ws = re.findall(r"\w+", t.lower())
    grams = [t[i:i+k] for i in range(0, max(len(t)-k, 0), step)]
    toks = [w.lower() for w in TOKEN_RE.findall(t)]
    return dict(ttr=len(set(ws))/len(ws) if ws else np.nan,
                kgram_unique=len(set(grams))/len(grams) if grams else np.nan,
                ascii_frac=sum(c.isascii() for c in t)/len(t) if t else np.nan,
                lat_frac=sum(bool(LAT_RE.match(c)) for c in t)/len(t) if t else np.nan,
                voc_ref=sum(w in VOC_REF for w in toks)/len(toks) if toks else np.nan,
                frasi_1k=1000*len(FRASE_RE.findall(t))/len(t) if t else np.nan,
                len_token=float(np.mean([len(w) for w in toks])) if toks else np.nan)

ref_deg = pd.DataFrame([degeneracy_flags(s) for s in ORIG_SEGMENTS]).mean()
KGRAM_SOGLIA = float(ref_deg["kgram_unique"]) - 0.15   # relativa al testo umano
LAT_SOGLIA   = 0.90                                    # umano ~0.99
VOC_SOGLIA   = 0.85                                    # vedi nota sulla circolarita'
print(f"riferimento umano: 10-grammi {ref_deg['kgram_unique']:.3f} | "
      f"latini {ref_deg['lat_frac']:.3f} | frasi/1k {ref_deg['frasi_1k']:.1f} | "
      f"len token {ref_deg['len_token']:.2f}")
print(f"soglie: ripetizione < {KGRAM_SOGLIA:.3f} | alfabeto < {LAT_SOGLIA:.2f} | "
      f"lessico < {VOC_SOGLIA:.2f}")
print("NOTA: la copertura di vocabolario del testo umano vale 1.000 per costruzione\n"
      "      (il vocabolario e' estratto da quello stesso corpus). La soglia 0.85 e'\n"
      "      quindi calibrata sulla bimodalita' osservata nei testi generati, non sul\n"
      "      valore umano, e va letta come criterio relativo.")

deg = pd.DataFrame([degeneracy_flags(TEXTS[i][:N_EFF]) for i in df["text_index"]],
                   index=df.index)
df = pd.concat([df, deg], axis=1)
df["deg_ripetizione"] = df["kgram_unique"] < KGRAM_SOGLIA
df["deg_alfabeto"]    = df["lat_frac"] < LAT_SOGLIA
df["deg_lessico"]     = df["voc_ref"] < VOC_SOGLIA
df["degenerato"]      = df["deg_ripetizione"] | df["deg_alfabeto"] | df["deg_lessico"]
df["testo_valido"]    = ~df["degenerato"] & df["usabile"]

_q = df.groupby("temperature").agg(
    n=("temperature", "size"),
    kgram=("kgram_unique", "mean"), lat=("lat_frac", "mean"), voc=("voc_ref", "mean"),
    ripetizione=("deg_ripetizione", "mean"), alfabeto=("deg_alfabeto", "mean"),
    lessico=("deg_lessico", "mean"), validi=("testo_valido", "sum")).round(3)
print("\nqualita' dei testi per temperatura:")
print(_q.to_string())
print(f"\nDOCUMENTI CON TESTO VALIDO: {int(df['testo_valido'].sum())} / {len(df)}")
print(df.pivot_table(index="model_short", columns="temperature", values="testo_valido",
                     aggfunc="sum").fillna(0).astype(int).to_string())
_solo3 = int((~df["deg_ripetizione"] & ~df["deg_alfabeto"] & df["deg_lessico"]).sum())
print(f"\ndocumenti che i primi due gate avrebbero lasciato passare e che il terzo\n"
      f"(lessico) intercetta: {_solo3}")
if df["testo_valido"].sum() < 0.2 * len(df):
    print("\n" + "!" * 78)
    print("ATTENZIONE: la grande maggioranza dei testi generati non e' prosa inglese.")
    print("Sotto una certa temperatura i modelli entrano in loop, sopra collassano in")
    print("un miscuglio di script, e una parte di quelli che sopravvivono ai primi due")
    print("controlli e' prosa solo nella prima meta' del documento. Le metriche di")
    print("Altmann calcolate su questi testi NON misurano struttura linguistica.")
    print("Le conclusioni vanno tratte SOLO dai documenti che superano i tre gate.")
    print("!" * 78)

# --- il testo umano e quelli generati hanno la stessa densita' di spazi bianchi? ---
def _densita(t):
    return dict(newline=t.count("\n") / len(t), capoverso=t.count("\n\n") / len(t),
                spazio=t.count(" ") / len(t))
_du = pd.DataFrame([_densita(s) for s in ORIG_SEGMENTS]).mean()
_sel = df.loc[df["usabile"], "text_index"]
_dl = pd.DataFrame([_densita(TEXTS[i][:N_EFF]) for i in _sel]).mean()
_cmp = pd.DataFrame({"umano": _du, "generati": _dl})
_cmp["rapporto"] = _cmp["generati"] / _cmp["umano"]
print("\ndensita' di spazi bianchi (frazione di caratteri):")
print(_cmp.round(4).to_string())
if not (0.7 < _cmp.loc["newline", "rapporto"] < 1.4):
    print("ATTENZIONE: densita' di a-capo molto diversa fra umano e generati; i tau non")
    print("            sono direttamente confrontabili. Controllare la pulizia del corpus.")

## 4. Selezione delle sequenze bersaglio

Altmann analizza per ogni libro 41 sequenze: vocali/consonanti, 20 al livello delle lettere
(spazio + 19 lettere più frequenti), 20 al livello delle parole (6 parole più frequenti,
7 sostantivi più frequenti, 7 parole con **frequenza appaiata** a quella dei sostantivi).

L'appaiamento in frequenza è la parte non negoziabile del disegno: senza, si confronterebbero
parole con statistica di conteggio diversa e il risultato sarebbe dominato dalla frequenza.

**Deviazione dichiarata.** Il paper usa il POS tagging per isolare i sostantivi. Qui, per restare
offline e deterministici, si usano due criteri combinati:
- **nome proprio** — la parola compare in maiuscolo a metà frase in più del `PROPER_CAP_RATIO`
  delle occorrenze (isola *prince, pierre, andrew, natasha* — esattamente gli outlier di Fig. 3);
- **contenuto** — non appartiene alla stoplist di parole funzione e ha almeno 4 caratteri.

Si costruiscono **due** insiemi di bersagli:
- `riferimento`: scelti una volta sul corpus umano e applicati a **tutti** i testi. Risponde a
  *"il modello riproduce la burstiness di queste parole?"*. Le parole assenti danno NaN, e la
  frazione di NaN è essa stessa un risultato.
- `auto`: riselezionati dentro ogni documento. Risponde a *"il modello produce
  comunque delle keyword bursty, quali che siano?"*.

In [ ]:
STOPWORDS = set('''a about above after again against all am an and any are as at be because been
before being below between both but by can cannot could did do does doing down during each few for
from further had has have having he her here hers herself him himself his how i if in into is it its
itself me more most my myself no nor not of off on once only or other ought our ours ourselves out
over own same she should so some such than that the their theirs them themselves then there these
they this those through to too under until up very was we were what when where which while who whom
why with would you your yours yourself yourselves said one two would shall may might must upon'''.split())

VOWELS = set("aeiou")
SENT_END = set(".!?")

def word_stats(text):
    '''frequenze dei tipi lessicali e rapporto di maiuscola NON a inizio frase.

    Il confine di frase e' dedotto dalla punteggiatura fra un token e il successivo (e dal
    capoverso). Nella versione precedente la bandiera sent_start veniva messa a False dopo
    il primo token e non tornava mai True: di fatto la condizione "non a inizio frase" non
    veniva mai applicata, e ogni maiuscola di inizio periodo contava come prova di nome
    proprio.'''
    freq, cap, tot = Counter(), Counter(), Counter()
    prev_end, primo = 0, True
    for m in TOKEN_RE.finditer(text):
        w = m.group(0); lw = w.lower()
        freq[lw] += 1
        gap = text[prev_end:m.start()]
        inizio_frase = primo or any(c in SENT_END for c in gap) or "\n\n" in gap
        if not inizio_frase:
            tot[lw] += 1
            if w[0].isupper():
                cap[lw] += 1
        prev_end, primo = m.end(), False
    return freq, {w: (cap[w] / tot[w] if tot[w] >= 3 else 0.0) for w in freq}

def select_targets(text, n_letters=N_LETTERS, n_func=N_FUNCTION,
                   n_key=N_KEYWORDS, n_matched=N_MATCHED):
    '''Ritorna un dict {livello: [(etichetta, tipo, chiave)]} secondo il disegno di Altmann.'''
    low = text.lower()
    letter_freq = Counter(c for c in low if c.isalpha() and c.isascii())
    letters = [c for c, _ in letter_freq.most_common(n_letters)]

    freq, capr = word_stats(text)
    ordered = [w for w, _ in freq.most_common() if len(w) >= 2]

    funcs = [w for w in ordered if w in STOPWORDS][:n_func]

    def is_proper(w):
        return capr.get(w, 0) >= PROPER_CAP_RATIO
    def is_key(w):
        return is_proper(w) or (w not in STOPWORDS and len(w) >= 4)
    keys = [w for w in ordered if is_key(w)][:n_key]

    # Controlli appaiati in frequenza: per ogni keyword la parola con frequenza piu' vicina
    # fra quelle non selezionate come keyword e non nomi propri, senza ripetizioni.
    # (Escludere l'intera classe is_key, come faceva la versione precedente, riduceva il
    # pool alle sole parole funzione e alle parole di 2-3 lettere.)
    kset = set(keys)
    pool = [w for w in ordered if w not in kset and not is_proper(w) and w not in funcs]
    matched, used = [], set()
    for k in keys[:n_matched]:
        fk = freq[k]
        cand = sorted((w for w in pool if w not in used),
                      key=lambda w: (abs(math.log((freq[w] + 1e-9) / (fk + 1e-9))), w))
        if cand:
            matched.append(cand[0]); used.add(cand[0])

    out = {
        "vc":      [("vocali", "vocali", VOWELS)],
        "lettere": [("spazio", "spazio", " ")] + [(c, "lettera", c) for c in letters],
        "parole":  ([(w, "funzione", w) for w in funcs] +
                    [(w, "keyword", w) for w in keys] +
                    [(w, "appaiata", w) for w in matched]),
    }
    return out, freq

REF_TARGETS, REF_FREQ = select_targets(BODY[:2_000_000])
for lev, items in REF_TARGETS.items():
    print(f"{lev:9s}: " + ", ".join(f"{lab}[{tp}]" for lab, tp, _ in items[:14]))
print()
kk = [l for l, t, _ in REF_TARGETS['parole'] if t == 'keyword']
mm = [l for l, t, _ in REF_TARGETS['parole'] if t == 'appaiata']
_ref = BODY[:2_000_000]
print("appaiamento in frequenza (per milione di caratteri) e verifica di ritrovabilita':")
for k, m in zip(kk, mm):
    nk, nm = len(pos_from_word(_ref, k)), len(pos_from_word(_ref, m))
    print(f"  {k:12s} {1e6*REF_FREQ[k]/len(_ref):8.1f} (match {nk:5d})   <->  "
          f"{m:12s} {1e6*REF_FREQ[m]/len(_ref):8.1f} (match {nm:5d})")
_rotte = [w for w in kk + mm if len(pos_from_word(_ref, w)) < 0.5 * REF_FREQ[w]]
if _rotte:
    print("\nATTENZIONE: parole contate dal tokenizzatore ma non ritrovate nel testo:", _rotte)
else:
    print("\nOK: ogni bersaglio viene ritrovato nel testo (nessun troncamento da accento).")

# quante occorrenze restano nella finestra effettiva di analisi?
_seg = BODY[:N_EFF]
_scarsi = [(w, len(pos_from_word(_seg, w))) for w in kk + mm
           if len(pos_from_word(_seg, w)) < MIN_EVENTS]
if _scarsi:
    print(f"\nNOTA: su {N_EFF:,} caratteri queste parole scendono sotto MIN_EVENTS={MIN_EVENTS}")
    print("      e daranno NaN anche nel testo umano:",
          ", ".join(f"{w} ({n})" for w, n in _scarsi))

In [ ]:
def targets_positions(text, targets):
    '''da {livello: [(label,tipo,chiave)]} a lista di (label, tipo, livello, posizioni)'''
    carr = build_char_array(text)
    out = []
    for lev, items in targets.items():
        for lab, tp, key in items:
            if lev == "vc":
                pos = pos_from_set(carr, key)
            elif tp == "spazio":
                pos = pos_from_char(carr, " ")
            elif tp == "lettera":
                pos = pos_from_char(carr, key)
            else:
                pos = pos_from_word(text, key)
            out.append((lab, tp, lev, pos))
    return out

## 5. Diagnostica di lunghezza finita

**È la cella che decide se i confronti successivi hanno senso.** Si prende il testo umano e si
misurano $\hat\gamma$ e $\sigma_\tau/\langle\tau\rangle$ su sottocampioni annidati di lunghezza
crescente. Se lo stimatore deriva con $N$ — e per una coda con $\mu<3$ deve derivare — allora il
valore misurato su 42 k caratteri **non** è confrontabile con quello del paper su 3.2 M, e la
differenza LLM-vs-Altmann non è un risultato sul modello.

Il numero da riportare in tesi: il valore umano a $N = N_{\rm eff}$, non quello del libro intero.

In [ ]:
FS_WORDS = [("prince", "keyword"), ("e", "lettera"), (" ", "spazio")]
fs_rows = []
if RUN_FINITE_SIZE:
    Ns = [5_000, 10_000, 20_000, 50_000, 100_000, 300_000, 1_000_000, len(BODY)]
    Ns = sorted({min(n, len(BODY)) for n in Ns})
    for N in Ns:
        seg = BODY[:N]
        carr = build_char_array(seg)
        for w, tp in FS_WORDS:
            pos = pos_from_char(carr, w) if tp in ("lettera", "spazio") else pos_from_word(seg, w)
            if len(pos) < MIN_EVENTS:
                continue
            s2 = transport_sigma2(pos, N)
            g, se = fit_gamma(LAGS, s2, *FIT_RANGE)
            b = burstiness_stats(interevent(pos))
            fs_rows.append(dict(N=N, parola=w, tipo=tp, n_eventi=len(pos),
                                gamma=g, gamma_se=se, **b))
        print(f"N={N:>9,} fatto", flush=True)
fs = pd.DataFrame(fs_rows)
if len(fs):
    fs.to_csv(OUT_DIR / "dati" / "finite_size.csv", index=False)
    print()
    print(fs.pivot_table(index="N", columns="parola", values="cv_tau").round(3).to_string())
    print("\n^ se questi numeri crescono con N, sigma_tau/<tau> NON e' confrontabile fra")
    print("  testi di lunghezza diversa: usare sempre i segmenti umani a N_EFF come riferimento.")

In [ ]:
if RUN_FINITE_SIZE and len(fs):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
    for w in fs["parola"].unique():
        g = fs[fs["parola"] == w].sort_values("N")
        lab = {"e": 'lettera "e"', " ": "spazio"}.get(w, f'"{w}"')
        axes[0].plot(g["N"], g["cv_tau"], marker="o", ms=5, label=lab)
        axes[1].errorbar(g["N"], g["gamma"], yerr=g["gamma_se"].fillna(0),
                         marker="o", ms=5, capsize=2.5, label=lab)
    for ax in axes:
        ax.set_xscale("log"); ax.set_xlabel("$N$ [caratteri]")
        ax.axvline(N_EFF, color="crimson", ls=":", lw=1.8)
        ax.legend(fontsize=8)
    axes[0].set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$")
    axes[0].set_title("A) la burstiness stimata deriva con la lunghezza")
    axes[1].set_ylabel(r"$\hat\gamma$"); axes[1].axhline(1, color="k", lw=0.8, alpha=.5)
    axes[1].set_title(r"B) $\hat\gamma$ e' molto piu' stabile")
    fig.suptitle("Diagnostica di lunghezza finita sul testo umano\n"
                 "(linea rossa = lunghezza dei testi generati)", fontweight="bold")
    fig.savefig(OUT_DIR / "figure" / "fig1_finite_size.png"); plt.show()

## 6. Analisi principale

Per ogni testo (segmenti umani + testi generati) e per ogni sequenza bersaglio si calcolano
$\hat\gamma$, $\hat\gamma_{A1}$, $\hat\gamma_{A2}$, $\sigma_\tau/\langle\tau\rangle$.

In [ ]:
def analyze_document(text, meta, targets, wordset_name, keep=()):
    N = min(len(text), N_EFF)
    t = text[:N]
    r = np.random.default_rng(seme(meta.get("label"), wordset_name))
    recs, curves = [], {}
    for lab, tp, lev, pos in targets_positions(t, targets):
        want = (lab in keep)
        rec, cur = analyze_sequence(pos, N, lab, lev, r, keep_curves=want)
        rec.update(tipo=tp, wordset=wordset_name, **meta)
        recs.append(rec)
        if want and cur is not None:
            curves[lab] = cur
    return recs, curves

ALL, CURVES = [], {}

_keys = [l for l, t, _ in REF_TARGETS["parole"] if t == "keyword"]
_kw = CURVE_KEYWORD if CURVE_KEYWORD in _keys else (_keys[0] if _keys else None)
KEEP_CURVES = [CURVE_LETTER] + ([_kw] if _kw else [])
if _kw != CURVE_KEYWORD:
    print(f"'{CURVE_KEYWORD}' non e' fra le keyword selezionate: per le curve uso '{_kw}'")
print("curve complete conservate per:", KEEP_CURVES)

# --- riferimenti umani ---
for i, seg in enumerate(ORIG_SEGMENTS):
    meta = dict(label=f"umano_{i}", gruppo="umano", modello="Guerra e pace",
                temperature=np.nan, sample_id=i, degenerato=False, params_B=np.nan)
    tg_self, _ = select_targets(seg)
    r1, c1 = analyze_document(seg, meta, REF_TARGETS, "riferimento",
                              keep=KEEP_CURVES if i == 0 else ())
    r2, _  = analyze_document(seg, meta, tg_self, "auto")
    ALL += r1 + r2
    if i == 0:
        CURVES["umano"] = c1
    print(".", end="", flush=True)
print(" riferimenti umani fatti")

In [ ]:
# --- testi generati ---
rep_done = set()
n_saltati = 0
for _, row in df.iterrows():
    txt = TEXTS[row["text_index"]]
    # avvertenza (A): si analizzano solo documenti lunghi almeno quanto la finestra comune,
    # altrimenti cv_tau e gamma non sono confrontabili con i segmenti umani
    if RICHIEDI_LUNGHEZZA_UGUALE:
        if len(txt) < N_EFF:
            n_saltati += 1
            continue
    elif len(txt) < LAG_MAX * 4:
        n_saltati += 1
        continue
    lab = f"{row['model_short']}_T{row['temperature']:.1f}_s{row['sample_id']}"
    meta = dict(label=lab, gruppo="llm", modello=row["model_short"],
                temperature=row["temperature"], sample_id=row["sample_id"],
                params_B=row["params_B"], degenerato=bool(row["degenerato"]),
                deg_ripetizione=bool(row["deg_ripetizione"]),
                deg_alfabeto=bool(row["deg_alfabeto"]),
                deg_lessico=bool(row["deg_lessico"]),
                kgram_unique=row["kgram_unique"], lat_frac=row["lat_frac"],
                voc_ref=row["voc_ref"],
                n_continuations=row.get("n_continuations"), file=row["file"])
    keep = ()
    kk = (row["model_short"], row["temperature"])
    if row["model_short"] == REPRESENTATIVE_MODEL and kk not in rep_done:
        keep = KEEP_CURVES; rep_done.add(kk)
    tg_self, _ = select_targets(txt[:N_EFF])
    r1, c1 = analyze_document(txt, meta, REF_TARGETS, "riferimento", keep=keep)
    r2, _  = analyze_document(txt, meta, tg_self, "auto")
    ALL += r1 + r2
    if keep and c1:
        CURVES[f"llm_T{row['temperature']:.1f}"] = c1
    print(".", end="", flush=True)

A = pd.DataFrame(ALL)
A.to_csv(OUT_DIR / "risultati_altmann.csv", index=False)
print(f"\n\ndocumenti generati analizzati: {len(df) - n_saltati}/{len(df)}"
      f"  (saltati perche' piu' corti di {N_EFF:,} caratteri: {n_saltati})")
print(f"record totali: {len(A):,}")
print(A.groupby(["gruppo", "wordset"])["stimabile"].agg(["size", "mean"]).round(3).to_string())

In [ ]:
# --- controllo dei null model: A1 deve dare gamma ~ 1 ovunque ---
chk = (A[A["stimabile"]].groupby(["gruppo", "livello"])[["gamma", "gamma_A1", "gamma_A2"]]
       .mean().round(3))
print("controllo A1 (atteso ~1.00 in ogni riga):")
print(chk.to_string())
bad = A[(A["stimabile"]) & (A["gamma_A1"] > 1.15)]
if len(bad):
    print(f"\nATTENZIONE: {len(bad)} sequenze con gamma_A1 > 1.15 — il null model non torna,")
    print("controllare il range di fit o il numero di eventi:")
    print(bad.groupby("livello")["n_eventi"].describe()[["count", "mean", "min"]].round(1).to_string())

In [ ]:
# --- copertura: quante parole di riferimento sopravvivono nei testi generati? ---
cov = (A[(A["wordset"] == "riferimento") & (A["tipo"].isin(["keyword", "appaiata"]))]
       .groupby(["gruppo", "modello", "temperature", "tipo"], dropna=False)["stimabile"].mean()
       .unstack().round(2))
print("frazione di parole di riferimento con abbastanza occorrenze per essere stimate:")
# dropna=False: senza, la riga del riferimento umano (temperature = NaN) sparirebbe
# dalla tabella e non ci sarebbe nulla con cui confrontare i valori degli LLM.
print(cov.to_string())
print("\nNota: una copertura bassa alle alte T non e' un problema tecnico, e' il risultato")
print("      (il modello ha smesso di parlare degli stessi referenti).")
cov.to_csv(OUT_DIR / "dati" / "copertura_parole.csv")

## 7. Figure principali

In [ ]:
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9"]
MARKERS = ["o", "s", "^", "D", "v", "P"]
LINESTY = ["-", "--", "-.", ":", (0, (3, 1, 1, 1)), (0, (5, 1))]
STYLE = {m: dict(color=PALETTE[i % len(PALETTE)], marker=MARKERS[i % len(MARKERS)],
                 linestyle=LINESTY[i % len(LINESTY)]) for i, m in enumerate(MODELS)}
TEMPS = sorted(t for t in df["temperature"].dropna().unique())
TCOL = {t: plt.cm.viridis(i / max(len(TEMPS) - 1, 1)) for i, t in enumerate(TEMPS)}

def cum_dist(tau):
    '''P(tau) cumulativa complementare, come nei pannelli A,C di Fig.2'''
    if len(tau) < 2:
        return np.array([]), np.array([])
    s = np.sort(tau)
    return s, 1.0 - np.arange(len(s)) / len(s)

def safe_loglog(ax, x, y, **kw):
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if m.sum() >= 2:
        ax.plot(x[m], y[m], **kw)

In [ ]:
# --- FIG 2: analogo diretto della Fig. 2 del paper ---
def panel_pair(axP, axS, cur, titolo):
    x, y = cum_dist(cur["tau"])
    safe_loglog(axP, x, y, color="k", lw=1.8, label="originale")
    axP.set_xscale("log"); axP.set_yscale("log")
    axP.set_xlabel(r"$\tau$"); axP.set_ylabel(r"$P(>\tau)$")
    cv = burstiness_stats(cur["tau"])["cv_tau"]
    axP.set_title(f"{titolo}\n" + r"$\sigma_\tau/\langle\tau\rangle$ = " + f"{cv:.2f}", fontsize=9.5)
    L = cur["lags"]
    for arr, kw in [(cur["s2"],   dict(color="k", lw=1.8, marker="o", ms=3, label="originale")),
                    (cur["s2_A1"], dict(color="#0072B2", lw=1.3, ls="--", label="A1 (0/1 mescolati)")),
                    (cur["s2_A2"], dict(color="#D55E00", lw=1.3, ls="-.", label=r"A2 ($\tau$ mescolati)"))]:
        safe_loglog(axS, L, arr, **kw)
    g, _ = fit_gamma(L, cur["s2"], *FIT_RANGE)
    ok = np.isfinite(cur["s2"]) & (cur["s2"] > 0)
    if ok.any():
        t0 = L[ok][0]; y0 = cur["s2"][ok][0]
        for gg, c in [(g, "grey"), (1.0, "lightgrey")]:
            axS.plot(L[ok], y0 * (L[ok] / t0) ** gg, color=c, lw=1, zorder=0)
    axS.set_xscale("log"); axS.set_yscale("log")
    axS.axvspan(*FIT_RANGE, color="grey", alpha=0.10)
    axS.set_xlabel("$t$"); axS.set_ylabel(r"$\sigma_X^2(t)$")
    axS.set_title(r"$\hat\gamma$ = " + f"{g:.2f}", fontsize=9.5)
    axS.legend(fontsize=6.5)

blocchi = [("umano", "testo umano")]
blocchi += [(k, f"LLM {k.split('_')[1]}") for k in sorted(CURVES) if k.startswith("llm_")][:2]
blocchi = [(k, t) for k, t in blocchi if k in CURVES and CURVES[k]]

if blocchi:
    ncol = 2 * len(KEEP_CURVES[:2])
    fig, axes = plt.subplots(len(blocchi), ncol,
                             figsize=(4 * ncol, 3.9 * len(blocchi)), squeeze=False)
    for i, (key, titolo) in enumerate(blocchi):
        cur = CURVES[key]
        for j, w in enumerate(KEEP_CURVES[:2]):
            if w in cur and cur[w] is not None:
                nome = f'lettera "{w}"' if len(w) == 1 else f'parola "{w}"' 
                panel_pair(axes[i][2*j], axes[i][2*j+1], cur[w], f"{titolo} — {nome}")
            else:
                for a in (axes[i][2*j], axes[i][2*j+1]):
                    a.text(.5, .5, f'"{w}"\nnon stimabile', ha="center", va="center",
                           transform=a.transAxes, fontsize=9, color="grey")
                    a.set_xticks([]); a.set_yticks([])
    fig.suptitle("Burstiness e correlazioni su livelli linguistici diversi\n"
                 "(analogo della Fig. 2 di Altmann et al.: lettera = correlata ma non bursty; "
                 "keyword = bursty)", fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure" / "fig2_burstiness_correlazione.png"); plt.show()

In [ ]:
# --- FIG 3: diagramma burstiness-correlazione (analogo Fig. 3) ---
def diagramma(ax, sub, colore_per, titolo, mark_deg=True):
    for tipo, mk, ms in [("lettera", "o", 34), ("spazio", "*", 90),
                         ("vocali", "X", 70), ("funzione", "s", 40),
                         ("appaiata", "v", 44), ("keyword", "D", 58)]:
        s = sub[sub["tipo"] == tipo]
        if not len(s):
            continue
        ax.scatter(s["cv_tau"], s["gamma"], s=ms, marker=mk, alpha=.75,
                   c=[colore_per(tipo)], edgecolors="k", linewidths=.4, label=tipo)
    if mark_deg and "degenerato" in sub:
        d = sub[sub["degenerato"] == True]
        if len(d):
            ax.scatter(d["cv_tau"], d["gamma"], s=110, facecolors="none",
                       edgecolors="crimson", linewidths=1.1, label="degenerato")
    ax.axhline(1, color="k", lw=1); ax.axvline(1, color="k", lw=1)
    ax.plot([1], [1], marker="*", ms=13, color="k")
    ax.annotate("Poisson", (1, 1), textcoords="offset points", xytext=(7, -13), fontsize=8)
    ax.set_xlabel(r"burstiness  $\sigma_\tau/\langle\tau\rangle$")
    ax.set_ylabel(r"correlazione  $\hat\gamma$")
    ax.set_title(titolo, fontsize=10)

COL = {"lettera": "#8c8c8c", "spazio": "#8c8c8c", "vocali": "#8c8c8c",
       "funzione": "#0072B2", "appaiata": "#009E73", "keyword": "#D55E00"}

S = A[(A["stimabile"]) & (A["wordset"] == "riferimento")]
umano = S[S["gruppo"] == "umano"]
llm   = S[S["gruppo"] == "llm"]

Tsel = [t for t in TEMPS if t in set(llm["temperature"])]
Tshow = [Tsel[len(Tsel)//4], Tsel[len(Tsel)//2], Tsel[3*len(Tsel)//4]] if len(Tsel) >= 4 else Tsel[:3]
Tshow = sorted(set(Tshow))

fig, axes = plt.subplots(1, 1 + len(Tshow), figsize=(5.2 * (1 + len(Tshow)), 4.8), squeeze=False)
diagramma(axes[0][0], umano, COL.__getitem__,
          f"testo umano ({N_EFF//1000}k caratteri)", mark_deg=False)
axes[0][0].legend(fontsize=7, loc="upper left")
for j, t in enumerate(Tshow):
    sub = llm[(llm["temperature"] == t) & (llm["modello"] == REPRESENTATIVE_MODEL)]
    if not len(sub):
        sub = llm[llm["temperature"] == t]
    diagramma(axes[0][j+1], sub, COL.__getitem__, f"generato, T = {t:.1f}")
xm = np.nanpercentile(S["cv_tau"], 99.5); ym = np.nanpercentile(S["gamma"], 99.5)
for ax in axes[0]:
    # il limite basso usa un percentile, non il minimo: i documenti degenerati
    # possono avere gamma << 1 e schiaccierebbero la scala
    ax.set_xlim(0, max(xm, 2))
    ax.set_ylim(min(0.85, np.nanpercentile(S["gamma"], 1)), max(ym, 1.8))
fig.suptitle("Diagramma burstiness-correlazione (analogo della Fig. 3 di Altmann et al.)\n"
             "atteso nel testo umano: lettere in basso a sinistra, keyword in alto a destra",
             fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "fig3_diagramma.png"); plt.show()

In [ ]:
# --- FIG 4: esponente locale gamma(t) (analogo Fig. 4) ---
fig, axes = plt.subplots(1, len(blocchi), figsize=(5.6 * max(len(blocchi), 1), 4.4), squeeze=False)
for i, (key, titolo) in enumerate(blocchi):
    ax = axes[0][i]
    for w, c in zip(KEEP_CURVES[:2], ["#0072B2", "#D55E00"]):
        cur = CURVES[key].get(w)
        if cur is None:
            continue
        x, gl = local_gamma(cur["lags"], cur["s2"])
        if len(x):
            nome = f'"{w}"' 
            ax.plot(x, gl, color=c, marker="o", ms=3, lw=1.3, label=nome)
    ax.axhline(1, color="k", lw=0.9)
    ax.axvspan(*FIT_RANGE, color="grey", alpha=0.10)
    ax.axvspan(100, 1000, color="orange", alpha=0.08)
    ax.set_xscale("log"); ax.set_xlabel("$t$ [caratteri]")
    ax.set_ylabel(r"$\hat\gamma(t)$"); ax.set_title(titolo, fontsize=10)
    ax.legend(fontsize=8)
fig.suptitle(r"Transizione da diffusione normale ad anomala, $\hat\gamma(t)="
             r"\Delta\log\sigma_X^2/\Delta\log t$"
             "\n(banda arancio = scala del paragrafo, dove Altmann colloca la transizione "
             "delle keyword)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "fig4_gamma_locale.png"); plt.show()

In [ ]:
# --- FIG 5: gamma e burstiness vs temperatura, per livello e per modello ---
def vs_T(ax, sub, col, ylabel, hline=None, ref=None):
    for m in MODELS:
        g = sub[sub["modello"] == m].groupby("temperature")[col].agg(["mean", "std", "count"])
        g = g[g["count"] > 0]
        if not len(g):
            continue
        ax.errorbar(g.index, g["mean"], yerr=g["std"].fillna(0), capsize=2.5,
                    ms=5, lw=1.4, label=m, **STYLE[m])
    if ref is not None and np.isfinite(ref):
        ax.axhline(ref, color="k", lw=1.8, label="umano (stessa lunghezza)")
    if hline is not None:
        ax.axhline(hline, color="grey", lw=0.9, ls=":")
    ax.set_xlabel("temperatura $T$"); ax.set_ylabel(ylabel)

fig, axes = plt.subplots(2, 3, figsize=(16.5, 8.6))
for j, (tipo, nome) in enumerate([("lettera", "lettere"), ("funzione", "parole funzione"),
                                  ("keyword", "keyword")]):
    lu = umano[umano["tipo"] == tipo]
    ll = llm[llm["tipo"] == tipo]
    vs_T(axes[0][j], ll, "gamma", r"$\hat\gamma$", hline=1.0, ref=lu["gamma"].mean())
    axes[0][j].set_title(f"{nome}: correlazione", fontsize=10)
    vs_T(axes[1][j], ll, "cv_tau", r"$\sigma_\tau/\langle\tau\rangle$",
         hline=1.0, ref=lu["cv_tau"].mean())
    axes[1][j].set_title(f"{nome}: burstiness", fontsize=10)
axes[0][0].legend(fontsize=7)
fig.suptitle("Correlazione e burstiness per livello linguistico, in funzione della temperatura\n"
             "(nero = testo umano misurato su segmenti della stessa lunghezza; "
             "punteggiato = Poisson)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "fig5_vs_temperatura.png"); plt.show()

## 8. Il test appaiato: keyword contro controlli a frequenza appaiata

È il test quantitativo del paper: nei 10 libri i sostantivi hanno $\hat\gamma$ maggiore in 56 casi
su 70 e $\sigma_\tau/\langle\tau\rangle$ maggiore in 55 su 70 rispetto alle parole appaiate in
frequenza. Qui lo stesso confronto viene fatto separatamente sul testo umano e su ogni cella
(modello, temperatura), con intervallo di Wilson sulla frazione.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z**2/n
    c = (p + z**2/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return (max(c-h, 0.0), min(c+h, 1.0))

def paired_test(sub, col):
    '''appaia ogni keyword col suo controllo dentro lo stesso documento'''
    kk = [l for l, t, _ in REF_TARGETS["parole"] if t == "keyword"]
    mm = [l for l, t, _ in REF_TARGETS["parole"] if t == "appaiata"]
    pairs = list(zip(kk, mm))
    wins = tot = 0
    diffs = []
    for lab, doc in sub.groupby("label"):
        d = doc.set_index("sequenza")[col].to_dict()
        for a, b in pairs:
            va, vb = d.get(a, np.nan), d.get(b, np.nan)
            if np.isfinite(va) and np.isfinite(vb):
                tot += 1; wins += int(va > vb); diffs.append(va - vb)
    lo, hi = wilson(wins, tot)
    p = np.nan
    if HAS_SCIPY and tot > 0:
        p = float(sps.binomtest(wins, tot, 0.5).pvalue)
    return dict(vittorie=wins, confronti=tot,
                frazione=wins/tot if tot else np.nan, ic_lo=lo, ic_hi=hi,
                diff_media=float(np.mean(diffs)) if diffs else np.nan, p=p)

pt_rows = []
for col in ["gamma", "cv_tau"]:
    r = paired_test(S[S["gruppo"] == "umano"], col)
    pt_rows.append(dict(gruppo="umano", modello="Guerra e pace",
                        temperature=np.nan, metrica=col, **r))
    for (m, t), sub in S[S["gruppo"] == "llm"].groupby(["modello", "temperature"]):
        r = paired_test(sub, col)
        pt_rows.append(dict(gruppo="llm", modello=m, temperature=t, metrica=col, **r))
PT = pd.DataFrame(pt_rows)
PT.to_csv(OUT_DIR / "dati" / "test_appaiato.csv", index=False)
print("RIFERIMENTO UMANO (atteso: frazione nettamente > 0.5, come il 56/70 del paper)")
print(PT[PT["gruppo"] == "umano"][["metrica", "vittorie", "confronti", "frazione",
                                    "ic_lo", "ic_hi", "p"]].round(3).to_string(index=False))

In [ ]:
if len(PT[PT["gruppo"] == "llm"]):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    for ax, col, nome in zip(axes, ["gamma", "cv_tau"],
                             [r"$\hat\gamma$", r"$\sigma_\tau/\langle\tau\rangle$"]):
        for m in MODELS:
            g = PT[(PT["metrica"] == col) & (PT["modello"] == m)].sort_values("temperature")
            g = g[g["confronti"] >= 5]
            if not len(g):
                continue
            # l'intervallo di Wilson non e' centrato sulla frazione osservata:
            # senza il clip matplotlib rifiuta yerr negativi
            yerr = np.clip(np.vstack([g["frazione"] - g["ic_lo"],
                                      g["ic_hi"] - g["frazione"]]), 0, None)
            ax.errorbar(g["temperature"], g["frazione"], yerr=yerr, capsize=3,
                        ms=5, lw=1.3, label=m, **STYLE[m])
        u = PT[(PT["metrica"] == col) & (PT["gruppo"] == "umano")]
        if len(u):
            ax.axhline(float(u["frazione"].iloc[0]), color="k", lw=1.8, label="umano")
            ax.axhspan(float(u["ic_lo"].iloc[0]), float(u["ic_hi"].iloc[0]),
                       color="k", alpha=.10)
        ax.axhline(0.5, color="grey", ls=":", lw=1.2)
        ax.set_ylim(0, 1); ax.set_xlabel("temperatura $T$")
        ax.set_ylabel(f"frazione di coppie con {nome} maggiore\nnella keyword")
        ax.set_title(f"{nome}: keyword vs controllo appaiato", fontsize=10)
        ax.legend(fontsize=7)
    fig.suptitle("Test appaiato keyword / controlli a frequenza appaiata\n"
                 "(0.5 punteggiato = nessuna differenza; barre = intervallo di Wilson)",
                 fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure" / "fig6_test_appaiato.png"); plt.show()

In [ ]:
# --- la dissociazione: gamma delle lettere riprodotto MA burstiness delle keyword no? ---
def cella(sub, tipo, col):
    v = sub[(sub["tipo"] == tipo)][col]
    return float(v.mean()) if len(v) else np.nan

diss_rows = []
u_g_let = cella(umano, "lettera", "gamma")
u_b_key = cella(umano, "keyword", "cv_tau")
u_g_key = cella(umano, "keyword", "gamma")
for (m, t), sub in llm.groupby(["modello", "temperature"]):
    diss_rows.append(dict(modello=m, temperature=t,
                          gamma_lettere=cella(sub, "lettera", "gamma"),
                          gamma_keyword=cella(sub, "keyword", "gamma"),
                          burst_keyword=cella(sub, "keyword", "cv_tau"),
                          burst_lettere=cella(sub, "lettera", "cv_tau"),
                          quota_burst_keyword=cella(sub, "keyword", "quota_burstiness")))
D = pd.DataFrame(diss_rows)
D["r_gamma_lettere"] = D["gamma_lettere"] / u_g_let
D["r_burst_keyword"] = D["burst_keyword"] / u_b_key
D.to_csv(OUT_DIR / "dati" / "dissociazione.csv", index=False)

fig, ax = plt.subplots(figsize=(7.6, 6.4))
for m in MODELS:
    g = D[D["modello"] == m]
    if not len(g):
        continue
    sc = ax.scatter(g["r_gamma_lettere"], g["r_burst_keyword"],
                    c=[TCOL.get(t, "grey") for t in g["temperature"]],
                    s=70, marker=STYLE[m]["marker"], edgecolors=STYLE[m]["color"],
                    linewidths=1.6, label=m)
ax.axhline(1, color="k", lw=1.2); ax.axvline(1, color="k", lw=1.2)
ax.plot([1], [1], marker="*", ms=16, color="k", zorder=5)
ax.annotate("testo umano", (1, 1), textcoords="offset points", xytext=(8, 8), fontsize=9)
ax.set_xlabel(r"$\hat\gamma$ lettere / valore umano")
ax.set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$ keyword / valore umano")
ax.set_title("La dissociazione\n"
             "in basso a destra = correlazioni riprodotte SENZA burstiness semantica\n"
             "(colore = temperatura)", fontsize=10)
ax.legend(fontsize=8)
sm = plt.cm.ScalarMappable(cmap="viridis",
                           norm=plt.Normalize(min(TEMPS), max(TEMPS)) if TEMPS else None)
if TEMPS:
    plt.colorbar(sm, ax=ax, label="temperatura")
fig.savefig(OUT_DIR / "figure" / "fig7_dissociazione.png"); plt.show()
print(D.round(3).to_string(index=False))

## 9. Shuffling M1 / M2

Il paper genera testi della stessa lunghezza con due manipolazioni non banali:

- **M1** — tiene fisse le posizioni degli spazi e ricolloca ogni token in un buco della sua misura.
  Distrugge i legami verso i livelli **sopra** le parole → deve **distruggere** le correlazioni.
- **M2** — ricodifica ogni tipo lessicale con una stringa casuale di pari lunghezza, applicata
  coerentemente a tutte le occorrenze. Rimescola i legami parola→lettera ma conserva la struttura
  sopra → deve **preservare** le correlazioni.

M2 che preserva è la verifica sperimentale dell'arbitrarietà del segno: le correlazioni non
dipendono da *quale* forma sia attaccata a quale referente.

In [ ]:
LETTERS = "abcdefghijklmnopqrstuvwxyz"

def shuffle_M1(text, r):
    '''tiene fissi gli spazi, permuta i token fra buchi della stessa lunghezza'''
    parts = re.split(r"(\s+)", text)
    idx = defaultdict(list)
    for i, p in enumerate(parts):
        if p and not p.isspace():
            idx[len(p)].append(i)
    out = list(parts)
    for L, ii in idx.items():
        vals = [parts[i] for i in ii]
        perm = r.permutation(len(vals))
        for i, k in zip(ii, perm):
            out[i] = vals[k]
    return "".join(out)

def shuffle_M2(text, r):
    '''ricodifica ogni tipo lessicale con una stringa casuale di pari lunghezza'''
    mapping = {}
    def rep(m):
        w = m.group(0); lw = w.lower()
        if lw not in mapping:
            mapping[lw] = "".join(r.choice(list(LETTERS), size=len(w)))
        return mapping[lw]
    return re.sub(r"[A-Za-z]+", rep, text)

SH = pd.DataFrame()
if RUN_SHUFFLES_M1M2:
    sh_rows = []
    def gamma_lettere(t, r):
        '''gamma medio sulle lettere piu' frequenti del testo dato'''
        N = min(len(t), N_EFF); tt = t[:N]
        carr = build_char_array(tt)
        gs = []
        for c, _ in Counter(x for x in tt.lower() if x.isalpha() and x.isascii()).most_common(10):
            pos = pos_from_char(carr, c)
            if len(pos) < MIN_EVENTS:
                continue
            g, _ = fit_gamma(LAGS, transport_sigma2(pos, N), *FIT_RANGE)
            if np.isfinite(g):
                gs.append(g)
        return float(np.mean(gs)) if gs else np.nan

    r = np.random.default_rng(RANDOM_SEED)
    for i, seg in enumerate(ORIG_SEGMENTS[:6]):
        sh_rows.append(dict(gruppo="umano", variante="originale", i=i, gamma=gamma_lettere(seg, r)))
        sh_rows.append(dict(gruppo="umano", variante="M1", i=i, gamma=gamma_lettere(shuffle_M1(seg, r), r)))
        sh_rows.append(dict(gruppo="umano", variante="M2", i=i, gamma=gamma_lettere(shuffle_M2(seg, r), r)))
        print(".", end="", flush=True)
    sel = (df[df["usabile"]].sort_values(["model_short", "temperature"])
             .groupby(["model_short", "temperature"], as_index=False).head(1))
    for _, row in sel.iterrows():
        t = TEXTS[row["text_index"]]
        base = dict(gruppo="llm", modello=row["model_short"], temperature=row["temperature"])
        sh_rows.append(dict(variante="originale", gamma=gamma_lettere(t, r), **base))
        sh_rows.append(dict(variante="M1", gamma=gamma_lettere(shuffle_M1(t[:N_EFF], r), r), **base))
        sh_rows.append(dict(variante="M2", gamma=gamma_lettere(shuffle_M2(t[:N_EFF], r), r), **base))
        print(".", end="", flush=True)
    SH = pd.DataFrame(sh_rows)
    SH.to_csv(OUT_DIR / "dati" / "shuffling_M1M2.csv", index=False)
    print("\n")
    print(SH.groupby(["gruppo", "variante"])["gamma"].agg(["mean", "std", "count"]).round(3).to_string())

In [ ]:
if RUN_SHUFFLES_M1M2 and len(SH):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    ax = axes[0]
    ordine = ["originale", "M1", "M2"]
    for j, gr in enumerate(["umano", "llm"]):
        g = SH[SH["gruppo"] == gr].groupby("variante")["gamma"].agg(["mean", "std"])
        g = g.reindex(ordine)
        ax.bar(np.arange(3) + .38*j - .19, g["mean"], .34, yerr=g["std"].fillna(0),
               capsize=4, label=gr, color=["#333333", "#0072B2"][j], alpha=.85)
    ax.set_xticks(range(3)); ax.set_xticklabels(ordine)
    ax.axhline(1, color="grey", ls=":", lw=1.2)
    ax.set_ylabel(r"$\hat\gamma$ medio sulle lettere")
    ax.set_title("M1 deve distruggere, M2 deve preservare", fontsize=10)
    ax.legend(fontsize=8)
    ax = axes[1]
    for m in MODELS:
        for v, ls in [("originale", "-"), ("M1", "--"), ("M2", "-.")]:
            g = (SH[(SH["modello"] == m) & (SH["variante"] == v)]
                 .sort_values("temperature"))
            if not len(g):
                continue
            ax.plot(g["temperature"], g["gamma"], ls=ls, marker=STYLE[m]["marker"],
                    ms=4, lw=1.2, color=STYLE[m]["color"],
                    label=f"{m} — {v}" if v != "originale" else m)
    ax.axhline(1, color="grey", ls=":", lw=1.2)
    ax.set_xlabel("temperatura $T$"); ax.set_ylabel(r"$\hat\gamma$ lettere")
    ax.set_title("per modello e temperatura", fontsize=10)
    ax.legend(fontsize=6, ncol=2)
    fig.suptitle("Shuffling M1 / M2 (Altmann et al., §Data Analysis of Shuffled Texts)",
                 fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure" / "fig8_shuffling.png"); plt.show()

## 10. Controlli di artefatto (avvertenze C e D)

Due controlli da riportare comunque nel capitolo metodologico, anche se passano:
la burstiness correla con la degenerazione? e col numero di riprese di generazione?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
key = llm[llm["tipo"] == "keyword"]
agg = (key.groupby("label")
          .agg(cv=("cv_tau", "mean"), gamma=("gamma", "mean"),
               kg=("kgram_unique", "first"), T=("temperature", "first"),
               nc=("n_continuations", "first")).reset_index())
ax = axes[0]
ax.scatter(agg["kg"], agg["cv"], c=[TCOL.get(t, "grey") for t in agg["T"]],
           s=45, edgecolors="k", linewidths=.4)
ax.axvline(KGRAM_SOGLIA, color="crimson", ls=":", lw=1.6)
ax.set_xlabel("unicita' dei 10-grammi (bassa = degenerato)")
ax.set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$ keyword")
if HAS_SCIPY:
    d = agg.dropna(subset=["kg", "cv"])
    if len(d) > 8:
        rho, pv = sps.spearmanr(d["kg"], d["cv"])
        esito = "nessun artefatto" if pv > .05 else "ATTENZIONE: correlato"
        ax.set_title(r"A) burstiness vs degenerazione   $\rho$=" + f"{rho:+.2f}, p={pv:.3f}\n{esito}",
                     fontsize=9.5)
ax = axes[1]
d = agg.dropna(subset=["nc"])
if len(d):
    ax.scatter(d["nc"], d["gamma"], c=[TCOL.get(t, "grey") for t in d["T"]],
               s=45, edgecolors="k", linewidths=.4)
    ax.set_xlabel("numero di riprese di generazione"); ax.set_ylabel(r"$\hat\gamma$ keyword")
    if HAS_SCIPY and len(d) > 8 and d["nc"].nunique() > 2:
        rho, pv = sps.spearmanr(d["nc"], d["gamma"])
        esito = "nessun artefatto" if pv > .05 else "ATTENZIONE: correlato"
        ax.set_title(r"B) $\hat\gamma$ vs riprese   $\rho$=" + f"{rho:+.2f}, p={pv:.3f}\n{esito}",
                     fontsize=9.5)
    print("riprese: mediana", d["nc"].median(), "| massimo", d["nc"].max(),
          "| documenti ricuciti", f"{(d['nc'] > 0).mean():.0%}")
else:
    ax.text(.5, .5, "campo n_continuations assente:\ngenerazione in una sola chiamata",
            ha="center", va="center", transform=ax.transAxes, fontsize=10, color="grey")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Controlli di artefatto", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "fig9_artefatti.png"); plt.show()

## 11. La domanda centrale: keyword bursty *quali che siano*, attorno a $T=1$

Le sezioni precedenti chiedono se il modello riproduce la burstiness **delle parole di Tolstoj**
(wordset `riferimento`). È una domanda severa: se il modello smette di parlare degli stessi
referenti, la risposta è NaN e non si impara nulla sul meccanismo.

Qui si pone la domanda debole ma ben posta: **il modello produce comunque delle keyword bursty,
qualunque esse siano?** Si usa il wordset `auto`, in cui i bersagli sono riselezionati dentro
ogni documento, e si confronta ogni keyword col proprio controllo a frequenza appaiata, **dentro
lo stesso documento**. Il confronto è quindi interno e non richiede alcuna sovrapposizione
lessicale col testo umano.

**Perché solo attorno a $T=1$.** La §3 mostra che fuori da una finestra stretta i testi non sono
prosa inglese: sotto $T\simeq0.9$ i modelli entrano in loop di ripetizione, sopra $T\simeq1.2$
l'alfabeto collassa. Entrambi i regimi producono numeri, e numeri ingannevoli:

- un testo in loop ha come parole di contenuto più frequenti quelle del ciclo ripetuto, quindi
  il wordset `auto` seleziona **proprio** quelle e misura una burstiness enorme e del tutto
  artefattuale — è il modo di fallimento a cui questa analisi è più esposta;
- un testo a script misto ha lettere ASCII sparse fra caratteri di altri alfabeti, e la loro
  spaziatura non ha alcun contenuto linguistico.

Per questo la finestra di temperatura **e** i due filtri di degenerazione sono applicati prima
dell'analisi, non come marcatura nelle figure. La cella seguente riporta anche il risultato
*senza* filtri, così si vede quanto pesano.

**Le due domande, separate:**

1. *interna* — dentro ogni documento, le keyword sono più bursty dei controlli appaiati in
   frequenza? È il test di Altmann, e non dipende dalla lunghezza né dal lessico.
2. *assoluta* — il livello di burstiness è quello umano? Qui la lunghezza conta, e il confronto
   è contro i segmenti umani da $N_{\rm eff}$ caratteri, mai contro il libro intero.

In [ ]:
# ── 11.1 selezione: finestra di temperatura e filtri di degenerazione ──────────
T_FINESTRA  = (0.9, 1.2)   # estremi inclusi; vedi tabella di qualita' della §3
SOLO_VALIDI = True         # scarta loop di ripetizione e collasso dell'alfabeto

def _b(s):
    """colonna booleana robusta ai NaN (le righe umane non hanno le bandiere LLM)"""
    return s.fillna(False).astype(bool)

def sottoinsieme(wordset="auto", solo_validi=SOLO_VALIDI, finestra=T_FINESTRA):
    """Righe da analizzare. NON filtra su `stimabile`: l'appaiamento keyword/controllo
    e' POSIZIONALE, quindi togliere qui le righe non stimabili sfaserebbe le coppie
    (la 3a keyword verrebbe appaiata al 4o controllo). La finitezza si verifica
    coppia per coppia dentro il test."""
    base = A["wordset"] == wordset
    hum  = base & (A["gruppo"] == "umano")
    llm  = base & (A["gruppo"] == "llm") & A["temperature"].between(*finestra)
    if solo_validi:
        llm = llm & ~_b(A["degenerato"])
    return A[hum | llm].copy()

AUTO = sottoinsieme()
U_AUTO = AUTO[AUTO["gruppo"] == "umano"]
L_AUTO = AUTO[AUTO["gruppo"] == "llm"]

print(f"finestra di temperatura: T in [{T_FINESTRA[0]}, {T_FINESTRA[1]}]"
      f" | filtri di degenerazione: {'ATTIVI' if SOLO_VALIDI else 'DISATTIVATI'}")
print(f"segmenti umani: {U_AUTO['label'].nunique()}")

_doc = (L_AUTO.groupby(["modello", "temperature"])["label"].nunique()
        .unstack(fill_value=0))
print("\ndocumenti generati che entrano nell'analisi:")
print(_doc.to_string() if len(_doc) else "  NESSUNO")
print(f"\ntotale documenti generati analizzati: {L_AUTO['label'].nunique()}")

# quanto costa il filtro? confronto con la stessa finestra senza filtri
_senza = sottoinsieme(solo_validi=False)
_ns = _senza[_senza["gruppo"] == "llm"]["label"].nunique()
print(f"(nella stessa finestra, senza filtri di degenerazione sarebbero {_ns})")
if L_AUTO["label"].nunique() == 0:
    print("\nATTENZIONE: nessun documento supera i filtri. Le celle seguenti non")
    print("            produrranno risultati sugli LLM. Allargare T_FINESTRA o")
    print("            disattivare SOLO_VALIDI serve solo a capire cosa succede,")
    print("            NON a ottenere un risultato: i testi degenerati danno")
    print("            burstiness artefattuale.")

In [ ]:
# ── 11.2 prima della burstiness: il modello produce keyword RICORRENTI? ───────
# Una parola puo' essere bursty solo se ricorre. Se in un documento nessuna parola di
# contenuto torna abbastanza spesso, la domanda sulla burstiness non si pone: non ci
# sono intervalli da misurare. Questa e' la diagnosi da guardare per prima, ed e' molto
# piu' robusta del test appaiato perche' non richiede che le sequenze siano stimabili.
def ricorrenza(sub):
    k = sub[sub["tipo"] == "keyword"]
    g = k.groupby("label")["n_eventi"]
    return pd.DataFrame({"top": g.max(), "mediana": g.median()})

_righe, _dist = [], {}
for nome, sub in ([("Guerra e pace", U_AUTO)] +
                  [(m, g) for m, g in L_AUTO.groupby("modello")]):
    if not len(sub):
        continue
    r = ricorrenza(sub)
    _dist[nome] = r["top"].values
    ev = sub[sub["tipo"] == "keyword"]["n_eventi"]
    _righe.append(dict(modello=nome, n_doc=len(r),
                       top_mediana=r["top"].median(), top_min=r["top"].min(),
                       top_max=r["top"].max(), occorrenze_mediane=ev.median(),
                       frac_sopra_soglia=(ev >= MIN_EVENTS).mean()))
RICORR = pd.DataFrame(_righe)
RICORR.to_csv(OUT_DIR / "dati" / "keyword_auto_T1_ricorrenza.csv", index=False)

print(f"Ricorrenza delle keyword auto-selezionate, su {N_EFF:,} caratteri "
      f"(MIN_EVENTS = {MIN_EVENTS})\n")
print(f"  {'':22s} {'doc':>4s} {'kw piu ricorrente':>18s} {'occorrenze':>11s} {'stimabili':>10s}")
print(f"  {'':22s} {'':>4s} {'(mediana [min-max])':>18s} {'(mediana)':>11s} {'':>10s}")
for _, x in RICORR.iterrows():
    print(f"  {x['modello']:22s} {int(x['n_doc']):4d} "
          f"{x['top_mediana']:8.0f} [{int(x['top_min']):3d}-{int(x['top_max']):3d}] "
          f"{x['occorrenze_mediane']:11.0f} {x['frac_sopra_soglia']:10.0%}")

_h = RICORR[RICORR["modello"] == "Guerra e pace"]
if len(_h):
    _hv = float(_h["top_mediana"].iloc[0])
    print(f"\n  Nel testo umano la keyword auto piu' ricorrente compare {_hv:.0f} volte "
          f"in {N_EFF:,} caratteri.")
    for _, x in RICORR[RICORR["modello"] != "Guerra e pace"].iterrows():
        rap = x["top_mediana"] / _hv if _hv else np.nan
        if rap < 0.4:
            print(f"  {x['modello']}: {x['top_mediana']:.0f} volte ({rap:.0%} dell'umano).")
            print(f"    -> il modello non stabilisce referenti persistenti. Non e' che le sue")
            print(f"       keyword non siano bursty: non ha keyword ricorrenti di cui misurare")
            print(f"       gli intervalli. Abbassare MIN_EVENTS produrrebbe confronti calcolati")
            print(f"       su pochi tau, nascondendo il risultato invece di misurarlo.")

if _dist:
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
    ax = axes[0]
    et = list(_dist.keys())
    bp = ax.boxplot([_dist[k] for k in et], tick_labels=[k.replace("-", "-\n") for k in et],
                    showmeans=True, widths=.55, patch_artist=True)
    for i, b in enumerate(bp["boxes"]):
        b.set(facecolor="#d9d9d9" if et[i] == "Guerra e pace" else "#cfe3f3", alpha=.85)
    for i, k in enumerate(et):
        v = _dist[k]
        ax.scatter(np.full(len(v), i + 1) + np.random.default_rng(3).normal(0, .05, len(v)),
                   v, s=18, color="k", alpha=.6, zorder=4)
    ax.axhline(MIN_EVENTS, color="crimson", ls=":", lw=1.6)
    ax.annotate(f"MIN_EVENTS = {MIN_EVENTS}", (0.55, MIN_EVENTS), fontsize=8,
                color="crimson", va="bottom")
    ax.set_ylabel("occorrenze della keyword più ricorrente")
    ax.set_title("A) il documento contiene un referente persistente?", fontsize=10)
    ax.tick_params(axis="x", labelsize=7.5)

    ax = axes[1]
    ax.bar(range(len(RICORR)), RICORR["frac_sopra_soglia"],
           color=["#8c8c8c" if m == "Guerra e pace" else "#0072B2"
                  for m in RICORR["modello"]], alpha=.85)
    ax.set_xticks(range(len(RICORR)))
    ax.set_xticklabels([m.replace("-", "-\n") for m in RICORR["modello"]], fontsize=7.5)
    ax.set_ylim(0, 1.05); ax.set_ylabel("frazione di keyword stimabili")
    ax.set_title(f"B) frazione con almeno {MIN_EVENTS} occorrenze", fontsize=10)
    fig.suptitle("Prima della burstiness: le keyword ricorrono?  "
                 f"(wordset auto, T in [{T_FINESTRA[0]}, {T_FINESTRA[1]}], "
                 f"N = {N_EFF:,} caratteri)", fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure" / "fig11_ricorrenza_keyword.png"); plt.show()

In [ ]:
# ── 11.3 domanda interna: keyword vs controllo appaiato, dentro ogni documento ──
def coppie_posizionali(doc):
    """(indice keyword, indice controllo) nell'ordine in cui select_targets li ha scelti.

    select_targets costruisce `matched[i]` come il controllo a frequenza piu' vicina
    a `keys[i]`, e i record vengono accodati in quell'ordine: l'ordine di riga dentro
    un (documento, wordset) E' l'appaiamento. sort_index() lo rende esplicito."""
    k = doc[doc["tipo"] == "keyword"].sort_index()
    m = doc[doc["tipo"] == "appaiata"].sort_index()
    n = min(len(k), len(m))
    return list(zip(k.index[:n], m.index[:n]))

def test_appaiato_auto(sub, col, frame=None):
    """frazione di coppie in cui la keyword supera il proprio controllo."""
    F = A if frame is None else frame
    vinte = tot = 0
    diffs, per_doc = [], []
    for _, doc in sub.groupby("label"):
        w = t = 0
        for ik, im in coppie_posizionali(doc):
            va, vb = F.at[ik, col], F.at[im, col]
            if np.isfinite(va) and np.isfinite(vb):
                t += 1; w += int(va > vb); diffs.append(va - vb)
        vinte += w; tot += t
        if t:
            per_doc.append(w / t)
    lo, hi = wilson(vinte, tot)
    p = float(sps.binomtest(vinte, tot, 0.5).pvalue) if (HAS_SCIPY and tot) else np.nan
    return dict(vittorie=vinte, confronti=tot, documenti=len(per_doc),
                frazione=vinte / tot if tot else np.nan, ic_lo=lo, ic_hi=hi, p=p,
                diff_media=float(np.mean(diffs)) if diffs else np.nan)

righe = []
r = test_appaiato_auto(U_AUTO, "cv_tau"); r.update(gruppo="umano", modello="Guerra e pace",
                                                   metrica="cv_tau"); righe.append(r)
r = test_appaiato_auto(U_AUTO, "gamma");  r.update(gruppo="umano", modello="Guerra e pace",
                                                   metrica="gamma");  righe.append(r)
for m, sub in L_AUTO.groupby("modello"):
    for col in ["cv_tau", "gamma"]:
        r = test_appaiato_auto(sub, col)
        r.update(gruppo="llm", modello=m, metrica=col)
        righe.append(r)
PT_AUTO = pd.DataFrame(righe)[["gruppo", "modello", "metrica", "documenti", "vittorie",
                               "confronti", "frazione", "ic_lo", "ic_hi", "p", "diff_media"]]
PT_AUTO.to_csv(OUT_DIR / "dati" / "keyword_auto_T1_appaiato.csv", index=False)

print("Le keyword auto-selezionate battono il proprio controllo a frequenza appaiata?")
print("(0.5 = nessuna differenza; il riferimento umano e' l'ultima riga di confronto)\n")
for met, nome in [("cv_tau", "BURSTINESS  sigma_tau/<tau>"), ("gamma", "CORRELAZIONE  gamma")]:
    print(nome)
    s = PT_AUTO[PT_AUTO["metrica"] == met]
    for _, x in s.iterrows():
        stelle = "" if not np.isfinite(x["p"]) else (
            " ***" if x["p"] < 1e-3 else " **" if x["p"] < 1e-2 else " *" if x["p"] < .05 else "  n.s.")
        print(f"  {x['modello']:22s} {int(x['vittorie']):4d}/{int(x['confronti']):4d} = "
              f"{x['frazione']:.2f} [{x['ic_lo']:.2f}, {x['ic_hi']:.2f}]"
              f"  su {int(x['documenti'])} doc  p={x['p']:.1e}{stelle}")
    print()

In [ ]:
# ── 11.4 domanda assoluta: il livello di burstiness e' quello umano? ───────────
# Si aggrega PRIMA dentro il documento (media sulle parole di quel tipo) e POI fra
# documenti: le parole dentro uno stesso testo non sono osservazioni indipendenti.
LIVELLI = [("lettera", "lettere"), ("funzione", "parole funzione"),
           ("appaiata", "controlli appaiati"), ("keyword", "keyword")]

def per_documento(sub, tipo, col):
    s = sub[sub["tipo"] == tipo]
    return s.groupby("label")[col].mean().dropna()

conf = []
for tipo, nome in LIVELLI:
    hu = per_documento(U_AUTO, tipo, "cv_tau")
    hg = per_documento(U_AUTO, tipo, "gamma")
    riga = dict(tipo=tipo, gruppo="umano", modello="Guerra e pace", n_doc=len(hu),
                cv_media=hu.mean(), cv_sd=hu.std(), cv_rapporto=1.0,
                gamma_media=hg.mean(), gamma_sd=hg.std(), gamma_rapporto=1.0, p_mw=np.nan)
    conf.append(riga)
    for m, sub in L_AUTO.groupby("modello"):
        lu = per_documento(sub, tipo, "cv_tau")
        lg = per_documento(sub, tipo, "gamma")
        p = np.nan
        if HAS_SCIPY and len(lu) >= 3 and len(hu) >= 3:
            p = float(sps.mannwhitneyu(lu, hu, alternative="two-sided").pvalue)
        conf.append(dict(tipo=tipo, gruppo="llm", modello=m, n_doc=len(lu),
                         cv_media=lu.mean(), cv_sd=lu.std(),
                         cv_rapporto=lu.mean() / hu.mean() if hu.mean() else np.nan,
                         gamma_media=lg.mean(), gamma_sd=lg.std(),
                         gamma_rapporto=lg.mean() / hg.mean() if hg.mean() else np.nan,
                         p_mw=p))
CONF_AUTO = pd.DataFrame(conf)
CONF_AUTO.to_csv(OUT_DIR / "dati" / "keyword_auto_T1_livelli.csv", index=False)

print("Burstiness sigma_tau/<tau> per livello, media fra documenti "
      f"(N = {N_EFF:,} caratteri per tutti)")
print("rapporto = LLM / umano; p = Mann-Whitney sui valori per documento contro l'umano\n")
for tipo, nome in LIVELLI:
    s = CONF_AUTO[CONF_AUTO["tipo"] == tipo]
    print(f"  {nome}")
    for _, x in s.iterrows():
        rap = "" if x["gruppo"] == "umano" else f"  rapporto={x['cv_rapporto']:.2f}"
        pp  = "" if not np.isfinite(x["p_mw"]) else f"  p={x['p_mw']:.3f}"
        print(f"    {x['modello']:22s} {x['cv_media']:.2f} +- {x['cv_sd']:.2f}"
              f"  ({int(x['n_doc'])} doc){rap}{pp}")
    print()

# la dissociazione, in una riga per modello
print("DISSOCIAZIONE (rapporto LLM/umano; 1.00 = indistinguibile dall'umano)")
print(f"  {'modello':22s} {'gamma lettere':>14s} {'gamma keyword':>14s} {'burst keyword':>14s}")
for m in CONF_AUTO.loc[CONF_AUTO['gruppo'] == 'llm', 'modello'].unique():
    g = CONF_AUTO[(CONF_AUTO["modello"] == m)]
    gl = g[g["tipo"] == "lettera"]["gamma_rapporto"].iloc[0]
    gk = g[g["tipo"] == "keyword"]["gamma_rapporto"].iloc[0]
    bk = g[g["tipo"] == "keyword"]["cv_rapporto"].iloc[0]
    print(f"  {m:22s} {gl:14.2f} {gk:14.2f} {bk:14.2f}")
print("\n  gamma lettere ~ 1 ma burst keyword << 1 = correlazioni riprodotte senza")
print("  burstiness semantica: il modello imita la statistica proiettata, non il meccanismo.")

In [ ]:
# ── 11.5 figura riassuntiva ────────────────────────────────────────────────────
_mods = list(CONF_AUTO.loc[CONF_AUTO["gruppo"] == "llm", "modello"].unique())
fig, axes = plt.subplots(1, 3, figsize=(17, 4.9))

# (A) test appaiato interno
ax = axes[0]
s_h = PT_AUTO[(PT_AUTO["gruppo"] == "umano") & (PT_AUTO["metrica"] == "cv_tau")]
s_l = PT_AUTO[(PT_AUTO["gruppo"] == "llm") & (PT_AUTO["metrica"] == "cv_tau")]
if len(s_l):
    y = np.arange(len(s_l))
    err = np.clip(np.vstack([s_l["frazione"] - s_l["ic_lo"],
                             s_l["ic_hi"] - s_l["frazione"]]), 0, None)
    ax.errorbar(s_l["frazione"], y, xerr=err, fmt="o", ms=8, capsize=4,
                color="#0072B2", lw=1.6)
    ax.set_yticks(y); ax.set_yticklabels(s_l["modello"], fontsize=8)
    ax.set_ylim(-0.6, len(s_l) - 0.4)
if len(s_h):
    ax.axvline(float(s_h["frazione"].iloc[0]), color="k", lw=1.8, label="umano")
    ax.axvspan(float(s_h["ic_lo"].iloc[0]), float(s_h["ic_hi"].iloc[0]),
               color="k", alpha=.10)
ax.axvline(0.5, color="grey", ls=":", lw=1.3, label="nessuna differenza")
ax.set_xlim(0, 1); ax.set_xlabel("frazione di coppie con keyword più bursty")
ax.set_title("A) domanda interna\nkeyword auto vs controllo appaiato", fontsize=10)
ax.legend(fontsize=7, loc="lower left")

# (B) burstiness assoluta per livello
ax = axes[1]
tipi = [t for t, _ in LIVELLI]
xs = np.arange(len(tipi))
hu = [CONF_AUTO[(CONF_AUTO["tipo"] == t) & (CONF_AUTO["gruppo"] == "umano")]["cv_media"].iloc[0]
      for t in tipi]
hs = [CONF_AUTO[(CONF_AUTO["tipo"] == t) & (CONF_AUTO["gruppo"] == "umano")]["cv_sd"].iloc[0]
      for t in tipi]
ax.errorbar(xs, hu, yerr=hs, fmt="s-", ms=9, lw=2.2, capsize=4, color="k", label="umano", zorder=5)
for i, m in enumerate(_mods):
    v = [CONF_AUTO[(CONF_AUTO["tipo"] == t) & (CONF_AUTO["modello"] == m)]["cv_media"].iloc[0]
         for t in tipi]
    e = [CONF_AUTO[(CONF_AUTO["tipo"] == t) & (CONF_AUTO["modello"] == m)]["cv_sd"].iloc[0]
         for t in tipi]
    st = STYLE.get(m, dict(color=PALETTE[i % len(PALETTE)], marker="o", linestyle="-"))
    ax.errorbar(xs + .06 * (i + 1), v, yerr=e, ms=6, lw=1.4, capsize=3, label=m, **st)
ax.axhline(1, color="grey", ls=":", lw=1.2)
ax.annotate("Poisson", (len(tipi) - .5, 1), fontsize=8, va="bottom", ha="right", color="grey")
ax.set_xticks(xs); ax.set_xticklabels([n for _, n in LIVELLI], fontsize=8, rotation=12)
ax.set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$")
ax.set_title("B) domanda assoluta\nlivello di burstiness per tipo di sequenza", fontsize=10)
ax.legend(fontsize=7)

# (C) distribuzione per documento della burstiness delle keyword
ax = axes[2]
dati, etich = [], []
hk = per_documento(U_AUTO, "keyword", "cv_tau")
if len(hk):
    dati.append(hk.values); etich.append(f"umano\n({len(hk)})")
for m in _mods:
    v = per_documento(L_AUTO[L_AUTO["modello"] == m], "keyword", "cv_tau")
    if len(v):
        dati.append(v.values); etich.append(f"{m}\n({len(v)})")
if dati:
    bp = ax.boxplot(dati, tick_labels=etich, showmeans=True, widths=.55, patch_artist=True)
    for i, box in enumerate(bp["boxes"]):
        box.set(facecolor="#d9d9d9" if i == 0 else "#cfe3f3", alpha=.85)
    for i, v in enumerate(dati):
        ax.scatter(np.full(len(v), i + 1) + np.random.default_rng(7).normal(0, .045, len(v)),
                   v, s=16, color="k", alpha=.55, zorder=4)
ax.axhline(1, color="grey", ls=":", lw=1.2)
ax.set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$ delle keyword")
ax.set_title("C) burstiness delle keyword per documento\n(punti = singoli documenti)", fontsize=10)
ax.tick_params(axis="x", labelsize=7.5)

fig.suptitle("Il modello produce keyword bursty, quali che siano?  "
             f"(wordset auto, T in [{T_FINESTRA[0]}, {T_FINESTRA[1]}], "
             f"{'testi degenerati esclusi' if SOLO_VALIDI else 'NESSUN filtro'}, "
             f"N = {N_EFF:,} caratteri)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "fig10_keyword_auto_T1.png"); plt.show()

# ── righe per la sintesi finale ────────────────────────────────────────────────
SINTESI_T1 = []
SINTESI_T1.append("KEYWORD BURSTY 'QUALI CHE SIANO' (wordset auto, "
                  f"T in [{T_FINESTRA[0]}, {T_FINESTRA[1]}]"
                  f"{', testi degenerati esclusi' if SOLO_VALIDI else ''})")
SINTESI_T1.append(f"  documenti generati analizzati: {L_AUTO['label'].nunique()}"
                  f" | segmenti umani: {U_AUTO['label'].nunique()}")
for met, nome in [("cv_tau", "burstiness"), ("gamma", "correlazione")]:
    for _, x in PT_AUTO[PT_AUTO["metrica"] == met].iterrows():
        SINTESI_T1.append(f"  {nome:13s} {x['modello']:22s} "
                          f"{int(x['vittorie']):4d}/{int(x['confronti']):4d} = {x['frazione']:.2f} "
                          f"[{x['ic_lo']:.2f}, {x['ic_hi']:.2f}]  p={x['p']:.1e}")
for m in _mods:
    g = CONF_AUTO[CONF_AUTO["modello"] == m]
    SINTESI_T1.append(
        f"  rapporto LLM/umano {m:22s} gamma lettere={g[g['tipo']=='lettera']['gamma_rapporto'].iloc[0]:.2f}"
        f"  burst keyword={g[g['tipo']=='keyword']['cv_rapporto'].iloc[0]:.2f}")
print("\n".join(SINTESI_T1))

## 12. Sintesi e archivio

In [ ]:
L = []
L.append("SINTESI — protocollo Altmann et al. (2012) sui testi generati")
L.append("=" * 68)
L.append(f"lunghezza di analisi: {N_EFF:,} caratteri per testo (prompt escluso)")
L.append(f"range di fit per gamma: t in {FIT_RANGE}  (versione conservativa: {FIT_RANGE_STRETTO})")
L.append(f"documenti: {len(df)} generati, {len(ORIG_SEGMENTS)} segmenti umani appaiati in lunghezza")
L.append("")
L.append("RIFERIMENTO UMANO (segmenti della stessa lunghezza dei testi generati)")
for tipo in ["lettera", "funzione", "appaiata", "keyword"]:
    s = umano[umano["tipo"] == tipo]
    if len(s):
        L.append(f"  {tipo:10s}  gamma = {s['gamma'].mean():.3f} +- {s['gamma'].std():.3f}"
                 f"   sigma_tau/<tau> = {s['cv_tau'].mean():.2f} +- {s['cv_tau'].std():.2f}"
                 f"   (gamma_A1 = {s['gamma_A1'].mean():.3f})")
if len(fs):
    f_full = fs[(fs['parola'] == 'prince')].sort_values('N')
    if len(f_full) >= 2:
        L.append(f"  deriva di lunghezza finita per 'prince': "
                 f"sigma_tau/<tau> = {f_full['cv_tau'].iloc[0]:.2f} a N={f_full['N'].iloc[0]:,} "
                 f"-> {f_full['cv_tau'].iloc[-1]:.2f} a N={f_full['N'].iloc[-1]:,}")
        L.append("  (per questo NON si confronta col 3.86 del paper, misurato sul libro intero)")
L.append("")
L.append("TEST APPAIATO keyword vs controlli a frequenza appaiata")
for _, r in PT[PT['gruppo'] == 'umano'].iterrows():
    L.append(f"  umano, {r['metrica']:8s}: {int(r['vittorie'])}/{int(r['confronti'])} "
             f"= {r['frazione']:.2f} [{r['ic_lo']:.2f}, {r['ic_hi']:.2f}]  p={r['p']:.1e}")
best = PT[(PT['gruppo'] == 'llm') & (PT['metrica'] == 'cv_tau') & (PT['confronti'] >= 5)]
if len(best):
    b = best.loc[best['frazione'].idxmax()]
    L.append(f"  miglior cella LLM (burstiness): {b['modello']} T={b['temperature']:.1f} -> "
             f"{b['frazione']:.2f} [{b['ic_lo']:.2f}, {b['ic_hi']:.2f}]")
L.append("")
L.append("TEMPERATURA PIU' VICINA AL TESTO UMANO, per metrica")
# Questa tabella e' calcolata su TUTTI i documenti, degenerati compresi: e' la piu'
# facile da citare fuori contesto ed e' quella su cui e' piu' facile sbagliarsi, per
# cui la riserva va stampata qui e non dieci righe piu' sotto.
_nv = int((~df["testo_valido"]).sum())
if _nv:
    L.append(f"  ATTENZIONE: media su tutti i documenti, inclusi i {_nv} degenerati.")
    L.append("  Alle alte T i testi sono a script misto: un gamma delle lettere vicino")
    L.append("  all'umano li' non indica somiglianza linguistica. Vedi QUALITA' DEI TESTI.")
for m in MODELS:
    g = llm[llm['modello'] == m]
    if not len(g):
        continue
    riga = [f"  {m:24s}"]
    for tipo, col, ref in [("lettera", "gamma", umano[umano['tipo'] == 'lettera']['gamma'].mean()),
                           ("keyword", "gamma", umano[umano['tipo'] == 'keyword']['gamma'].mean()),
                           ("keyword", "cv_tau", umano[umano['tipo'] == 'keyword']['cv_tau'].mean())]:
        s = g[g['tipo'] == tipo].groupby('temperature')[col].mean().dropna()
        riga.append(f"{tipo[:3]}/{col}: T={s.sub(ref).abs().idxmin():.1f}" if len(s) else f"{tipo[:3]}/{col}: n.d.")
    L.append("  ".join(riga))
L.append("")
if df["degenerato"].any():
    L.append(f"QUALITA' DEI TESTI: {int(df['testo_valido'].sum())}/{len(df)} documenti sono "
             f"prosa inglese non degenerata.")
    L.append(f"  in loop di ripetizione (10-grammi < {KGRAM_SOGLIA:.2f}): "
             f"{int(df['deg_ripetizione'].sum())}  -> burstiness artefattuale")
    L.append(f"  alfabeto collassato (latini < {LAT_SOGLIA:.2f}):        "
             f"{int(df['deg_alfabeto'].sum())}  -> gamma delle lettere privo di significato")
    L.append(f"  lessico non inglese (copertura < {VOC_SOGLIA:.2f}): "
             f"{int(df['deg_lessico'].sum())}  -> spesso prosa solo nella prima meta'")
    _val = df.groupby("temperature")["testo_valido"].sum()
    _val = _val[_val > 0]
    L.append("  finestra di temperature con documenti validi: "
             + (", ".join(f"T={t:.1f} ({int(n)})" for t, n in _val.items()) if len(_val)
                else "NESSUNA"))
cnt = df.groupby(["model_short", "temperature"]).size()
if cnt.min() < cnt.max():
    L.append("NOTA: numerosita' disuguale fra celle: le deviazioni standard non sono "
             "confrontabili fra modelli.")
if "SINTESI_T1" in globals() and SINTESI_T1:
    L.append("")
    L += SINTESI_T1
summary = "\n".join(L)
(OUT_DIR / "sintesi.txt").write_text(summary, encoding="utf-8")
print(summary)

In [ ]:
readme = f'''Analisi Altmann et al. (PNAS 2012) sui testi generati da LLM

File
----
risultati_altmann.csv        una riga per (documento x sequenza bersaglio)
sintesi.txt                  sintesi testuale
dati/finite_size.csv         deriva degli stimatori con la lunghezza del testo
dati/copertura_parole.csv    frazione di parole di riferimento stimabili
dati/test_appaiato.csv       keyword vs controlli appaiati, con intervalli di Wilson
dati/dissociazione.csv       gamma lettere e burstiness keyword, normalizzati sull'umano
dati/shuffling_M1M2.csv      esiti degli shuffling M1/M2
dati/keyword_auto_T1_appaiato.csv  keyword auto vs controlli, dentro ogni documento
dati/keyword_auto_T1_livelli.csv   burstiness per livello, LLM vs umano, attorno a T=1
dati/keyword_auto_T1_ricorrenza.csv ricorrenza delle keyword auto (diagnosi primaria)
figure/fig1..fig11 *.png     figure

Colonne principali di risultati_altmann.csv
-------------------------------------------
sequenza          etichetta della sequenza binaria (lettera, parola, "vocali", "spazio")
tipo              lettera | spazio | vocali | funzione | keyword | appaiata
livello           vc | lettere | parole  (livello nella gerarchia di Fig. 1 del paper)
wordset           riferimento (parole scelte sul testo umano) | auto (scelte nel documento)
n_eventi          occorrenze; sotto MIN_EVENTS={MIN_EVENTS} la sequenza non viene stimata
gamma             esponente da sigma^2_X(t) ~ t^gamma, fit su t in {FIT_RANGE}
gamma_se          errore standard della pendenza
gamma_stretto     stesso fit sul range conservativo {FIT_RANGE_STRETTO} (1% della lunghezza)
gamma_A1          null model A1 (0/1 mescolati): deve valere ~1
gamma_A2          null model A2 (tau mescolati): preserva p(tau), distrugge C_tau(k)
quota_burstiness  (gamma_A2 - 1)/(gamma - 1): quota di correlazione dovuta alla burstiness
cv_tau            sigma_tau/<tau>, indicatore di burstiness (1 = Poisson)
B_goh             (sigma_tau - <tau>)/(sigma_tau + <tau>), burstiness di Goh-Barabasi
degenerato        True se il documento e' in loop di ripetizione: burstiness artefattuale

Come leggere i risultati
------------------------
1. Controllare fig1: se sigma_tau/<tau> deriva con N, confrontare SOLO con l'umano a N_EFF.
2. Controllare che gamma_A1 ~ 1 ovunque: e' la validazione della pipeline.
3. fig3 e' l'analogo della Fig. 3 del paper: nel testo umano le lettere stanno in basso a
   sinistra (correlate ma non bursty) e le keyword in alto a destra.
4. fig7 e' la domanda: in basso a destra = correlazioni riprodotte senza burstiness semantica.

Parametri: N_EFF={{N_EFF}}, MIN_EVENTS={MIN_EVENTS}, N_NULL_REPS={N_NULL_REPS}, seed={RANDOM_SEED}
'''
(OUT_DIR / "LEGGIMI.txt").write_text(readme.replace("{N_EFF}", str(N_EFF)), encoding="utf-8")

zip_path = shutil.make_archive(ZIP_NAME, "zip", root_dir=".", base_dir=str(OUT_DIR))
print("archivio:", Path(zip_path).resolve(), f"({Path(zip_path).stat().st_size/1e6:.1f} MB)")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT_DIR.parent))